<a href="https://colab.research.google.com/github/puwaphat/Getting-Started-Streamlit/blob/main/PEA_AI_2026_VSPP_Contest_Notebook_v4_LARGE_BEST_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PEA AI Contest 2026 — VSPP Forecasting v4 LARGE
## Automatic Best-Model Selection from `dataset_2_large`

This version is developed directly from the previous VSPP Forecasting v3 notebook.

### v4 improvements
- Connects automatically to the Google Drive folder `dataset_2_large`.
- Benchmarks Baseline, Classical ML, Machine Learning and Deep Learning fairly.
- Every Deep Learning experiment runs **50 epochs**.
- Uses **two validation views**: Contest-Mimic Gap + Temporal/Future.
- Selects the winner with a balanced forecasting score:
  - **RMSE 50%**
  - **MAE 30%**
  - **WMAPE 20%**
- Validation weighting:
  - **Gap 70%**
  - **Temporal 30%**
- The final champion may be an individual model, a Top-3 Ensemble, a Multi-Seed Deep model, or a Deep+Profile+Neighbor blend.
- The final test prediction uses the **actual validation winner**, not a pre-selected architecture.
- Classification Accuracy / Recall / F1 stay only in the auxiliary ramp-event task.

> **Contest principle:** No model is called “best” because of its name. Out-of-sample evidence decides the winner.


In [1]:
# ============================================================
# 0. Install/import and experiment configuration
# ============================================================
import os, sys, math, random, copy, warnings, itertools, time, json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

SEED = 230
EPOCHS = 50
BATCH_SIZE = 32
TEMPORAL_VALID_DAYS = 7
GAP_VALID_PATTERNS = 2
MAPE_EPS_MW = 0.05
RAMP_THRESHOLD_FRACTION = 0.05

# dataset_2_large — Google Drive
DATASET_FOLDER_NAME = "dataset_2_large"
DATASET_FOLDER_ID = "1pbcNEGGopIxLj0xUKDszsCeixR0l6fkw"
DATASET_FOLDER_URL = "https://drive.google.com/drive/folders/1pbcNEGGopIxLj0xUKDszsCeixR0l6fkw"

# Exact CSV IDs inside dataset_2_large, used only as fallback.
DRIVE_FILE_IDS = {
    "vspp_train.csv": "13hJSswBMXmSapNVz4apP-TFk77XC-OWv",
    "vspp_test.csv":  "1h7JTZM4BhHThwQf0W7KvTlK5X_8fMf1S",
    "weather.csv":    "1LAisjpmJ8TMrpK6BFYbdJ3zApwhvkxvf",
    "vspp_mw.csv":    "1slwnkYTaOpQMXNeFwtCgYpM2awrycyQ0",
}

# Optional explicit path, e.g.
# "/content/drive/MyDrive/PEA_AI_Contest/dataset_2_large"
DATA_DIR_OVERRIDE = None

# Validation weighting
GAP_WEIGHT = 0.70
TEMPORAL_WEIGHT = 0.30

# Forecasting metric weighting
RMSE_WEIGHT = 0.50
MAE_WEIGHT = 0.30
WMAPE_WEIGHT = 0.20

RUN_CLASSICAL = True
RUN_ARCHITECTURES = True
RUN_LOSS_BENCHMARK = True
RUN_OPTIMIZER_BENCHMARK = True
RUN_AUXILIARY_BENCHMARK = True
RUN_MULTI_SEED_FINAL = True
MULTI_SEEDS = [230, 42, 2026]

RUN_OPTUNA = False
ALLOW_TEST_LABEL_AUDIT = False

np.random.seed(SEED)
random.seed(SEED)

print("PEA AI Contest 2026 — VSPP Forecasting v4 LARGE")
print("Deep-learning epochs per run:", EPOCHS)
print("Dataset folder:", DATASET_FOLDER_NAME)
print("Test-label audit enabled:", ALLOW_TEST_LABEL_AUDIT)


PEA AI Contest 2026 — VSPP Forecasting v4 LARGE
Deep-learning epochs per run: 50
Dataset folder: dataset_2_large
Test-label audit enabled: False


## 1. Data path — `dataset_2_large`

The notebook is configured for the Google Drive folder:

`dataset_2_large`

Folder ID: `1pbcNEGGopIxLj0xUKDszsCeixR0l6fkw`

Expected files:
- `vspp_train.csv`
- `vspp_test.csv`
- `weather.csv`
- `vspp_mw.csv` — optional audit/reference only

The resolver mounts Google Drive in Colab, searches for `dataset_2_large`, and falls back to the exact Drive file IDs if necessary.


In [2]:
# ============================================================
# 1. Resolve dataset_2_large from Google Drive / local runtime
# ============================================================
def _contains_required_csvs(folder):
    folder = Path(folder)
    return (
        folder.exists()
        and (folder / "vspp_train.csv").exists()
        and (folder / "vspp_test.csv").exists()
        and (folder / "weather.csv").exists()
    )

def _find_named_folder(root, folder_name, max_depth=8):
    root = Path(root)
    if not root.exists():
        return None

    root_depth = len(root.parts)
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.parts) - root_depth

        if depth > max_depth:
            dirs[:] = []
            continue

        if current_path.name == folder_name and _contains_required_csvs(current_path):
            return current_path

        if folder_name in dirs:
            candidate = current_path / folder_name
            if _contains_required_csvs(candidate):
                return candidate

    return None

def _download_drive_fallback(target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    try:
        import gdown
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown

    for filename, file_id in DRIVE_FILE_IDS.items():
        out = target_dir / filename
        if out.exists() and out.stat().st_size > 0:
            continue

        print("Downloading fallback:", filename)
        result = gdown.download(id=file_id, output=str(out), quiet=False)

        if result is None and filename != "vspp_mw.csv":
            raise RuntimeError(
                f"Could not download {filename}. "
                "Add dataset_2_large to My Drive or set DATA_DIR_OVERRIDE."
            )

    return target_dir

DATA_DIR = None

# 1) Explicit path
if DATA_DIR_OVERRIDE:
    candidate = Path(DATA_DIR_OVERRIDE)
    if not _contains_required_csvs(candidate):
        raise FileNotFoundError(
            f"DATA_DIR_OVERRIDE does not contain required CSV files: {candidate}"
        )
    DATA_DIR = candidate

# 2) Colab Drive mount
if DATA_DIR is None and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    fast_candidates = [
        Path("/content/drive/MyDrive") / DATASET_FOLDER_NAME,
        Path("/content/drive/MyDrive/PEA_AI_Contest") / DATASET_FOLDER_NAME,
        Path("/content/drive/MyDrive/PEA AI Contest 2026") / DATASET_FOLDER_NAME,
    ]

    for candidate in fast_candidates:
        if _contains_required_csvs(candidate):
            DATA_DIR = candidate
            break

    if DATA_DIR is None:
        for root in [
            Path("/content/drive/MyDrive"),
            Path("/content/drive/Shareddrives"),
        ]:
            found = _find_named_folder(root, DATASET_FOLDER_NAME)
            if found is not None:
                DATA_DIR = found
                break

# 3) Local runtime
if DATA_DIR is None:
    for candidate in [
        Path.cwd() / DATASET_FOLDER_NAME,
        Path.cwd(),
        Path("/content") / DATASET_FOLDER_NAME,
        Path("/mnt/data") / DATASET_FOLDER_NAME,
        Path("/mnt/data"),
    ]:
        if _contains_required_csvs(candidate):
            DATA_DIR = candidate
            break

# 4) Direct file-ID fallback
if DATA_DIR is None:
    fallback_dir = (
        Path("/content") / DATASET_FOLDER_NAME
        if Path("/content").exists()
        else Path.cwd() / DATASET_FOLDER_NAME
    )
    DATA_DIR = _download_drive_fallback(fallback_dir)

assert _contains_required_csvs(DATA_DIR), f"Dataset not found: {DATA_DIR}"

TRAIN_FILE = DATA_DIR / "vspp_train.csv"
TEST_FILE = DATA_DIR / "vspp_test.csv"
WEATHER_FILE = DATA_DIR / "weather.csv"
MW_FILE = DATA_DIR / "vspp_mw.csv"
if not MW_FILE.exists():
    MW_FILE = None

print("Resolved DATA_DIR:", DATA_DIR)
print("TRAIN:", TRAIN_FILE)
print("TEST :", TEST_FILE)
print("WEATHER:", WEATHER_FILE)
print("MW reference:", MW_FILE)


Mounted at /content/drive


Downloading...
From: https://drive.google.com/uc?id=13hJSswBMXmSapNVz4apP-TFk77XC-OWv
To: /content/dataset_2_large/vspp_train.csv
100%|██████████| 1.58M/1.58M [00:00<00:00, 12.2MB/s]


Downloading...
From: https://drive.google.com/uc?id=1h7JTZM4BhHThwQf0W7KvTlK5X_8fMf1S
To: /content/dataset_2_large/vspp_test.csv
100%|██████████| 63.9k/63.9k [00:00<00:00, 2.56MB/s]


Downloading...
From: https://drive.google.com/uc?id=1LAisjpmJ8TMrpK6BFYbdJ3zApwhvkxvf
To: /content/dataset_2_large/weather.csv
100%|██████████| 2.98M/2.98M [00:00<00:00, 22.9MB/s]


Downloading...
From: https://drive.google.com/uc?id=1slwnkYTaOpQMXNeFwtCgYpM2awrycyQ0
To: /content/dataset_2_large/vspp_mw.csv
100%|██████████| 1.97M/1.97M [00:00<00:00, 14.0MB/s]

Resolved DATA_DIR: /content/dataset_2_large
TRAIN: /content/dataset_2_large/vspp_train.csv
TEST : /content/dataset_2_large/vspp_test.csv
WEATHER: /content/dataset_2_large/weather.csv
MW reference: /content/dataset_2_large/vspp_mw.csv


In [4]:

# ============================================================
# 2. Load and audit raw data
# ============================================================
train_all_raw = pd.read_csv(TRAIN_FILE)
test_raw = pd.read_csv(TEST_FILE)
weather_raw = pd.read_csv(WEATHER_FILE)
mw_reference = pd.read_csv(MW_FILE) if MW_FILE is not None else None

required_train = {"generation_date","generation_time","name","mw"}
required_test_base = {"generation_date","generation_time","name"}
required_weather = {"name","forecast_datetime","lagging_hour","tc","rh","rain","swdown","ws10m"}

assert required_train.issubset(train_all_raw.columns)
assert required_test_base.issubset(test_raw.columns)
assert required_weather.issubset(weather_raw.columns)

print("train:", train_all_raw.shape)
print("test :", test_raw.shape)
print("weather:", weather_raw.shape)
print("mw:", mw_reference.shape)
print("train dates:", train_all_raw["generation_date"].nunique())
print("sites:", sorted(train_all_raw["name"].unique()))
print("test contains MW label:", "mw" in test_raw.columns)
print("complete MW reference loaded:", mw_reference is not None)


train: (42129, 5)
test : (1719, 5)
weather: (34983, 9)
mw: (52461, 5)
train dates: 49
sites: ['ZS11', 'ZS12', 'ZS13', 'ZS14', 'ZS22', 'ZS23', 'ZW11', 'ZW12', 'ZW13']
test contains MW label: True
complete MW reference loaded: True


In [5]:

# ============================================================
# 3. Time features
# ============================================================
def add_time(df):
    d = df.copy()
    d["dt"] = pd.to_datetime(d["generation_date"] + " " + d["generation_time"])
    d["q"] = (d["dt"].dt.hour * 4 + d["dt"].dt.minute // 15).astype(int)
    d["dow"] = d["dt"].dt.dayofweek.astype(int)
    d["doy"] = d["dt"].dt.dayofyear.astype(int)
    d["hour"] = d["dt"].dt.hour + d["dt"].dt.minute/60.0

    d["is_solar"] = d["name"].str.startswith("ZS").astype(int)
    d["is_wind"] = d["name"].str.startswith("ZW").astype(int)

    d["sin_q"] = np.sin(2*np.pi*d["q"]/96)
    d["cos_q"] = np.cos(2*np.pi*d["q"]/96)
    d["sin_dow"] = np.sin(2*np.pi*d["dow"]/7)
    d["cos_dow"] = np.cos(2*np.pi*d["dow"]/7)
    d["sin_doy"] = np.sin(2*np.pi*d["doy"]/365.25)
    d["cos_doy"] = np.cos(2*np.pi*d["doy"]/365.25)
    return d

train_all = add_time(train_all_raw)
test = add_time(test_raw)

print(train_all["dt"].min(), "to", train_all["dt"].max())
print("test dates:", sorted(test["generation_date"].unique()))


2026-06-01 00:00:00 to 2026-07-30 23:45:00
test dates: ['2026-06-08', '2026-06-15']



## 2. Two validation views

### A. Temporal validation
The last 7 known training dates are held out. This answers:

> “Will the model generalize to future dates using only information available before the forecast?”

### B. Contest-mimic gap validation
The timestamps in the supplied test data show isolated missing interior dates. Without looking at test `mw`, we can use the **timestamp pattern only** to create pseudo-test dates inside the known training period.

This answers:

> “Can the model reconstruct an isolated missing generation day when known days exist around it?”

The final model should be strong on **both**, but the contest-mimic score receives more weight for leaderboard selection.


In [6]:

# ============================================================
# 4. Build temporal + contest-mimic validation dates
# ============================================================
all_dates = pd.to_datetime(sorted(train_all["generation_date"].unique()))
test_dates = pd.to_datetime(sorted(test["generation_date"].unique()))

temporal_dates = list(all_dates[-TEMPORAL_VALID_DAYS:])
temporal_set = set(temporal_dates)
train_date_set = set(all_dates)

# Derive the pattern of gaps from test timestamps WITHOUT using test MW labels.
test_offsets = sorted(set((test_dates - test_dates.min()).days.tolist()))
if not test_offsets:
    test_offsets = [0]

def eligible_anchor(anchor):
    pattern_dates = [anchor + pd.Timedelta(days=int(o)) for o in test_offsets]
    if not all(d in train_date_set for d in pattern_dates):
        return False
    if any(d in temporal_set for d in pattern_dates):
        return False

    # Require previous and next known calendar day around each pseudo-gap.
    for d in pattern_dates:
        if (d - pd.Timedelta(days=1)) not in train_date_set:
            return False
        if (d + pd.Timedelta(days=1)) not in train_date_set:
            return False

    # Keep enough distance from temporal validation.
    if max(pattern_dates) > min(temporal_dates) - pd.Timedelta(days=8):
        return False
    return True

anchors = [d for d in all_dates if eligible_anchor(d)]
if len(anchors) < GAP_VALID_PATTERNS:
    warnings.warn("Not enough pattern-matched anchors; falling back to eligible isolated dates.")
    anchors = [
        d for d in all_dates
        if d not in temporal_set
        and (d-pd.Timedelta(days=1)) in train_date_set
        and (d+pd.Timedelta(days=1)) in train_date_set
        and d <= min(temporal_dates)-pd.Timedelta(days=8)
    ]

# Deterministic spread across the eligible period.
qs = np.linspace(1/(GAP_VALID_PATTERNS+1), GAP_VALID_PATTERNS/(GAP_VALID_PATTERNS+1), GAP_VALID_PATTERNS)
selected_anchors = [anchors[int(round((len(anchors)-1)*q))] for q in qs]

gap_dates = sorted(set(
    a + pd.Timedelta(days=int(o))
    for a in selected_anchors
    for o in test_offsets
))
gap_dates = [d for d in gap_dates if d not in temporal_set]

temporal_date_str = {d.strftime("%Y-%m-%d") for d in temporal_dates}
gap_date_str = {d.strftime("%Y-%m-%d") for d in gap_dates}

temporal_valid = train_all[train_all["generation_date"].isin(temporal_date_str)].copy()
gap_valid = train_all[train_all["generation_date"].isin(gap_date_str)].copy()

# One core training set for fair architecture comparison.
heldout_dates = temporal_date_str | gap_date_str
core_train = train_all[~train_all["generation_date"].isin(heldout_dates)].copy()

print("Temporal validation dates:", sorted(temporal_date_str))
print("Gap validation dates:", sorted(gap_date_str))
print("Core train:", core_train.shape)
print("Gap validation:", gap_valid.shape)
print("Temporal validation:", temporal_valid.shape)


Temporal validation dates: ['2026-07-24', '2026-07-25', '2026-07-26', '2026-07-27', '2026-07-28', '2026-07-29', '2026-07-30']
Gap validation dates: ['2026-06-25', '2026-06-27', '2026-07-02', '2026-07-04']
Core train: (32652, 18)
Gap validation: (3438, 18)
Temporal validation: (6039, 18)


In [7]:

# ============================================================
# 5. Weather: latest available forecast per site/target hour
# ============================================================
weather = weather_raw.copy()

# Preserve the local wall-clock represented in +07:00 strings.
weather["dt"] = pd.to_datetime(weather["forecast_datetime"]).dt.tz_localize(None)

weather = (
    weather.sort_values(["name","dt","lagging_hour"])
           .drop_duplicates(["name","dt"], keep="first")
)

WEATHER_COLS = ["tc","rh","rain","swdown","ws10m"]

start = min(train_all["dt"].min(), test["dt"].min()).floor("h")
end = max(train_all["dt"].max(), test["dt"].max()).ceil("h")

parts = []
for site, g in weather.groupby("name"):
    g = g[["dt"] + WEATHER_COLS].set_index("dt").sort_index()
    idx = pd.date_range(start, end, freq="15min")
    gi = g.reindex(idx).interpolate(method="time").ffill().bfill()
    gi["dt"] = idx
    gi["name"] = site
    parts.append(gi.reset_index(drop=True))

weather15 = pd.concat(parts, ignore_index=True)
print("15-minute weather:", weather15.shape)


15-minute weather: (51849, 7)



## 3. Feature engineering

New additions beyond the original notebook:

- `profile_std`, `profile_p10`, `profile_p90`.
- lag/next context at 1, 2, 3, 7 and 14 days.
- explicit **missingness flags**, so a missing neighboring day is not silently treated as a real profile value.
- weather change features.
- wind-speed squared/cubed terms to help represent a nonlinear wind power curve.
- solar-radiation × solar-site interaction.


In [8]:

# ============================================================
# 6. Leakage-safe context features
# ============================================================
LAG_DAYS = [1,2,3,7,14]

def context_features(target, source, use_future_context):
    d = target.copy().merge(weather15, on=["name","dt"], how="left")

    site_stats = source.groupby("name")["mw"].agg(
        site_p99=lambda x: max(float(x.quantile(0.99)), 0.05),
        site_mean="mean",
        site_std="std",
        site_min="min",
        site_max="max",
    ).reset_index()
    d = d.merge(site_stats, on="name", how="left")

    prof = source.groupby(["name","q"])["mw"].agg(
        profile_mean="mean",
        profile_median="median",
        profile_std="std",
        profile_p10=lambda x: x.quantile(0.10),
        profile_p90=lambda x: x.quantile(0.90),
    ).reset_index()
    d = d.merge(prof, on=["name","q"], how="left")

    lookup = source.set_index(["name","dt"])["mw"]

    for day in LAG_DAYS:
        prev_idx = pd.MultiIndex.from_arrays(
            [d["name"], d["dt"] - pd.Timedelta(days=day)]
        )
        prev = lookup.reindex(prev_idx).to_numpy()
        d[f"prev_{day}d"] = prev
        d[f"prev_{day}d_missing"] = pd.isna(prev).astype(int)

        if use_future_context:
            next_idx = pd.MultiIndex.from_arrays(
                [d["name"], d["dt"] + pd.Timedelta(days=day)]
            )
            nxt = lookup.reindex(next_idx).to_numpy()
        else:
            nxt = np.full(len(d), np.nan)

        d[f"next_{day}d"] = nxt
        d[f"next_{day}d_missing"] = pd.isna(nxt).astype(int)
        d[f"bidir_{day}d"] = d[[f"prev_{day}d",f"next_{day}d"]].mean(axis=1)

    d["lag123_mean"] = d[["prev_1d","prev_2d","prev_3d"]].mean(axis=1)
    d["bidir123_mean"] = d[["bidir_1d","bidir_2d","bidir_3d"]].mean(axis=1)

    # Physical/weather interactions
    d["ws10m_sq"] = d["ws10m"]**2
    d["ws10m_cu"] = d["ws10m"]**3
    d["solar_swdown"] = d["is_solar"] * d["swdown"]
    d["wind_ws10m"] = d["is_wind"] * d["ws10m"]

    d = d.sort_values(["name","dt"]).copy()
    for c in WEATHER_COLS:
        d[f"d_{c}"] = d.groupby("name")[c].diff().fillna(0.0)

    return d

def leakage_free_training_features(source, use_future_context):
    # Each target day is feature-engineered from all OTHER core-training days.
    # This prevents its own MW values leaking into its profile/context features.
    parts = []
    for day in sorted(source["generation_date"].unique()):
        target_day = source[source["generation_date"] == day].copy()
        src = source[source["generation_date"] != day].copy()
        parts.append(context_features(target_day, src, use_future_context))
    return pd.concat(parts, ignore_index=True)

# Core training learns the same context style as contest reconstruction.
train_ctx = leakage_free_training_features(core_train, use_future_context=True)

# Strictly no held-out target values are included in these sources.
gap_ctx = context_features(gap_valid, core_train, use_future_context=True)
temporal_ctx = context_features(temporal_valid, core_train, use_future_context=False)

print(train_ctx.shape, gap_ctx.shape, temporal_ctx.shape)


(32652, 69) (3438, 69) (6039, 69)



## 4. Regression measurement

Primary regression metrics:

- **RMSE** — primary large-error metric.
- **MAE** — average MW error.
- **MAPE with epsilon** — transparent zero-safe percentage metric.
- **Active MAPE** — MAPE only when `|MW| >= 0.05`.
- **WMAPE** — robust total percentage error.
- **sMAPE** — symmetric percentage error.
- **R²** — explained variance.
- **MASE** — performance relative to the seasonal-profile baseline.

Because solar MW is often exactly zero, ordinary raw MAPE alone should never be the sole model-selection metric.


In [9]:

# ============================================================
# 7. Metrics
# ============================================================
def regression_metrics(y, pred, naive=None, eps=MAPE_EPS_MW):
    y = np.asarray(y, float)
    pred = np.asarray(pred, float)
    err = y - pred

    rmse = float(np.sqrt(np.mean(err**2)))
    mae = float(np.mean(np.abs(err)))
    active = np.abs(y) >= eps

    out = {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE_eps_%": float(np.mean(np.abs(err)/np.maximum(np.abs(y),eps))*100),
        "Active_MAPE_%": (
            float(np.mean(np.abs(err[active])/np.abs(y[active]))*100)
            if active.any() else np.nan
        ),
        "WMAPE_%": float(np.sum(np.abs(err))/max(np.sum(np.abs(y)),1e-12)*100),
        "sMAPE_%": float(np.mean(
            2*np.abs(err)/np.maximum(np.abs(y)+np.abs(pred),eps)
        )*100),
        "R2": float(r2_score(y,pred)),
    }

    if naive is not None:
        naive = np.asarray(naive,float)
        denom = np.mean(np.abs(y-naive))
        out["MASE"] = float(mae/denom) if denom > 1e-12 else np.nan
    else:
        out["MASE"] = np.nan

    return out

def classification_metrics(y_true,y_pred):
    return {
        "Accuracy": float(accuracy_score(y_true,y_pred)),
        "Precision_macro": float(precision_score(y_true,y_pred,average="macro",zero_division=0)),
        "Recall_macro": float(recall_score(y_true,y_pred,average="macro",zero_division=0)),
        "F1_macro": float(f1_score(y_true,y_pred,average="macro",zero_division=0)),
    }

def site_macro_metrics(df,pred):
    z = df[["name","mw"]].copy()
    z["pred"] = np.asarray(pred)
    rows = []
    for site,g in z.groupby("name"):
        m = regression_metrics(g["mw"],g["pred"])
        rows.append(m)
    return pd.DataFrame(rows).mean(numeric_only=True).add_prefix("Macro_").to_dict()


In [10]:

# ============================================================
# 8. Physical constraints
# ============================================================
def physical_postprocess(pred, df, source):
    p = np.asarray(pred,float).copy()

    limits = source.groupby("name")["mw"].agg(
        lo=lambda x: float(x.quantile(0.001)),
        hi=lambda x: float(x.quantile(0.999)),
    ).to_dict("index")

    for i,site in enumerate(df["name"]):
        lo = limits[site]["lo"] - 0.05
        hi = limits[site]["hi"]*1.08 + 0.02
        if site.startswith("ZS"):
            lo = 0.0
        p[i] = np.clip(p[i],lo,hi)

    # If the learned solar profile is exactly zero, force physical zero.
    solar_zero = (
        df["name"].str.startswith("ZS").to_numpy()
        & (df["profile_median"].fillna(0).to_numpy() <= 0)
    )
    p[solar_zero] = 0.0
    return p



## 5. Classical benchmark first

Deep learning does not receive special treatment. It must beat:

**Seasonal profile → persistence → Ridge/ElasticNet → Random Forest → XGBoost → LightGBM**

Every model uses the same core-training set and the same two validation views.


In [11]:
# ============================================================
# 9. Classical / ML benchmark
# ============================================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor

NUM_FEATURES = [
    "profile_mean","profile_median","profile_std","profile_p10","profile_p90",
    "site_mean","site_std",
    *[f"prev_{d}d" for d in LAG_DAYS],
    *[f"next_{d}d" for d in LAG_DAYS],
    *[f"bidir_{d}d" for d in LAG_DAYS],
    *[f"prev_{d}d_missing" for d in LAG_DAYS],
    *[f"next_{d}d_missing" for d in LAG_DAYS],
    "lag123_mean","bidir123_mean",
    *WEATHER_COLS,
    *[f"d_{c}" for c in WEATHER_COLS],
    "ws10m_sq","ws10m_cu","solar_swdown","wind_ws10m",
    "sin_q","cos_q","sin_dow","cos_dow","sin_doy","cos_doy",
    "q","dow","doy","is_solar","is_wind"
]
CAT_FEATURES = ["name"]

BASELINE_MODELS = ["Seasonal_Profile_Median", "Previous_Day_Persistence"]
CLASSICAL_MODEL_NAMES = ["Ridge", "ElasticNet", "RandomForest", "XGBoost", "LightGBM"]

def prepare_tabular(train_df, val_df):
    Xtr = train_df[NUM_FEATURES + CAT_FEATURES].copy()
    Xv = val_df[NUM_FEATURES + CAT_FEATURES].copy()

    for c in NUM_FEATURES:
        fill = train_df[c].median()
        fill = 0.0 if pd.isna(fill) else float(fill)
        Xtr[c] = Xtr[c].fillna(fill)
        Xv[c] = Xv[c].fillna(fill)
    return Xtr, Xv

def build_classical_model(name):
    """Return a fresh model for validation or final full-data refit."""
    if name in ["Ridge", "ElasticNet"]:
        prep_linear = ColumnTransformer([
            ("num", StandardScaler(), NUM_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
        ])

        if name == "Ridge":
            return make_pipeline(prep_linear, Ridge(alpha=0.1))

        return make_pipeline(
            prep_linear,
            ElasticNet(
                alpha=0.001,
                l1_ratio=0.2,
                max_iter=10000,
                random_state=SEED
            )
        )

    prep_tree = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ], remainder="passthrough")

    if name == "RandomForest":
        return make_pipeline(
            prep_tree,
            RandomForestRegressor(
                n_estimators=350,
                min_samples_leaf=3,
                max_features=0.8,
                n_jobs=-1,
                random_state=SEED
            )
        )

    if name == "XGBoost":
        from xgboost import XGBRegressor
        return make_pipeline(
            prep_tree,
            XGBRegressor(
                n_estimators=700,
                max_depth=5,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.90,
                reg_alpha=0.05,
                reg_lambda=1.5,
                objective="reg:squarederror",
                n_jobs=-1,
                random_state=SEED
            )
        )

    if name == "LightGBM":
        from lightgbm import LGBMRegressor
        return make_pipeline(
            prep_tree,
            LGBMRegressor(
                n_estimators=700,
                num_leaves=31,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.90,
                reg_alpha=0.05,
                reg_lambda=1.0,
                n_jobs=-1,
                verbosity=-1,
                random_state=SEED
            )
        )

    raise ValueError(f"Unknown classical model: {name}")

Xtr_gap, Xgap = prepare_tabular(train_ctx, gap_ctx)
Xtr_temp, Xtemp = prepare_tabular(train_ctx, temporal_ctx)
ytr = train_ctx["mw"].to_numpy()
ygap = gap_ctx["mw"].to_numpy()
ytemp = temporal_ctx["mw"].to_numpy()

prediction_pool = {"gap": {}, "temporal": {}}
rows = []
validation_model_objects = {}

def register_predictions(name, gap_pred, temp_pred, family):
    gap_pred = physical_postprocess(gap_pred, gap_ctx, core_train)
    temp_pred = physical_postprocess(temp_pred, temporal_ctx, core_train)

    prediction_pool["gap"][name] = gap_pred
    prediction_pool["temporal"][name] = temp_pred

    gm = regression_metrics(
        ygap, gap_pred,
        naive=gap_ctx["profile_median"].fillna(0).to_numpy()
    )
    tm = regression_metrics(
        ytemp, temp_pred,
        naive=temporal_ctx["profile_median"].fillna(0).to_numpy()
    )

    row = {"Model": name, "Family": family}
    row.update({f"Gap_{k}": v for k, v in gm.items()})
    row.update({f"Temporal_{k}": v for k, v in tm.items()})
    row.update({f"Gap_{k}": v for k, v in site_macro_metrics(gap_ctx, gap_pred).items()})
    row.update({f"Temporal_{k}": v for k, v in site_macro_metrics(temporal_ctx, temp_pred).items()})
    rows.append(row)

register_predictions(
    "Seasonal_Profile_Median",
    gap_ctx["profile_median"].fillna(0).to_numpy(),
    temporal_ctx["profile_median"].fillna(0).to_numpy(),
    "Baseline"
)

register_predictions(
    "Previous_Day_Persistence",
    gap_ctx["prev_1d"].fillna(gap_ctx["profile_median"]).to_numpy(),
    temporal_ctx["prev_1d"].fillna(temporal_ctx["profile_median"]).to_numpy(),
    "Baseline"
)

if RUN_CLASSICAL:
    for model_name in CLASSICAL_MODEL_NAMES:
        try:
            print("Training classical model:", model_name)
            model = build_classical_model(model_name)
            model.fit(Xtr_gap, ytr)
            validation_model_objects[model_name] = model

            family = (
                "Machine Learning"
                if model_name in ["RandomForest", "XGBoost", "LightGBM"]
                else "Classical"
            )

            register_predictions(
                model_name,
                model.predict(Xgap),
                model.predict(Xtemp),
                family
            )
        except Exception as e:
            print(f"{model_name} skipped:", e)

classical_table = pd.DataFrame(rows).set_index("Model")
display(
    classical_table[
        [
            "Family",
            "Gap_RMSE","Gap_MAE","Gap_WMAPE_%",
            "Temporal_RMSE","Temporal_MAE","Temporal_WMAPE_%"
        ]
    ]
    .sort_values("Gap_RMSE")
    .round(4)
)


Training classical model: Ridge
Training classical model: ElasticNet
Training classical model: RandomForest
Training classical model: XGBoost
Training classical model: LightGBM


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Family,Gap_RMSE,Gap_MAE,Gap_WMAPE_%,Temporal_RMSE,Temporal_MAE,Temporal_WMAPE_%
Model,,,,,,,
RandomForest,Machine Learning,0.8901,0.4490,37.4460,0.9632,0.4662,43.8755
XGBoost,Machine Learning,0.9005,0.4553,37.9667,0.8740,0.4341,40.8569
LightGBM,Machine Learning,0.9060,0.4543,37.8877,0.8886,0.4426,41.6618
Ridge,Classical,0.9096,0.4605,38.4051,0.8396,0.4324,40.7022
ElasticNet,Classical,0.9231,0.4759,39.6865,0.9352,0.4668,43.9333
Seasonal_Profile_Median,Baseline,1.0063,0.4825,40.2415,0.9262,0.4414,41.5487
Previous_Day_Persistence,Baseline,1.2390,0.5787,48.2623,0.9766,0.4647,43.7406



## 6. Deep-learning data

Each site-day is a 96-step sequence.

The regression head predicts a **site-scaled residual around the median profile**.

The auxiliary classification heads predict:
- binary: **large ramp / no large ramp** → BCE.
- 3-class: **down / stable / up** → CCE or hinge.

This is the correct place to use accuracy, recall and F1.


In [12]:

# ============================================================
# 10. Torch imports + sequence construction
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as tud

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:",DEVICE)

GEN_FEATURES = [
    "profile_median","profile_mean","profile_std","profile_p10","profile_p90",
    *[f"prev_{d}d" for d in LAG_DAYS],
    *[f"next_{d}d" for d in LAG_DAYS],
    *[f"bidir_{d}d" for d in LAG_DAYS],
    "lag123_mean","bidir123_mean",
]
MISSING_FEATURES = [
    *[f"prev_{d}d_missing" for d in LAG_DAYS],
    *[f"next_{d}d_missing" for d in LAG_DAYS],
]
WEATHER_FEATURES = [
    *WEATHER_COLS,
    *[f"d_{c}" for c in WEATHER_COLS],
    "ws10m_sq","ws10m_cu","solar_swdown","wind_ws10m"
]
TIME_FEATURES = [
    "sin_q","cos_q","sin_dow","cos_dow","sin_doy","cos_doy",
    "is_solar","is_wind"
]
SEQ_FEATURES = GEN_FEATURES + MISSING_FEATURES + WEATHER_FEATURES + TIME_FEATURES

sites = sorted(train_all["name"].unique())
site_to_idx = {s:i for i,s in enumerate(sites)}

def make_day_samples(df):
    samples = []

    for (date,site),g in df.groupby(["generation_date","name"]):
        h = pd.DataFrame({"q":np.arange(96)}).merge(g,on="q",how="left")

        h["profile_median"] = h["profile_median"].fillna(0)
        fallback = h["profile_median"]

        for c in GEN_FEATURES:
            h[c] = h[c].fillna(fallback).fillna(0)

        for c in MISSING_FEATURES:
            h[c] = h[c].fillna(1)

        for c in WEATHER_FEATURES:
            h[c] = h[c].interpolate().ffill().bfill().fillna(0)

        h["sin_q"] = np.sin(2*np.pi*h["q"]/96)
        h["cos_q"] = np.cos(2*np.pi*h["q"]/96)
        for c in ["sin_dow","cos_dow","sin_doy","cos_doy","is_solar","is_wind"]:
            h[c] = h[c].ffill().bfill().fillna(0)

        X = h[SEQ_FEATURES].to_numpy(np.float32)
        site_onehot = np.zeros((96,len(sites)),np.float32)
        site_onehot[:,site_to_idx[site]] = 1
        X = np.concatenate([X,site_onehot],axis=1)

        mask = (~h["mw"].isna()).to_numpy(np.float32)
        base = h["profile_median"].to_numpy(np.float32)
        y = h["mw"].fillna(h["profile_median"]).to_numpy(np.float32)

        scale = float(max(g["site_p99"].iloc[0],0.05))
        residual = (y-base)/scale

        # Ramp labels; valid only where current and previous target are both observed.
        diff = np.r_[0.0,np.diff(y)]
        threshold = max(scale*RAMP_THRESHOLD_FRACTION,0.02)
        ramp_class = np.ones(96,dtype=np.int64)
        ramp_class[diff < -threshold] = 0
        ramp_class[diff >  threshold] = 2
        ramp_binary = (np.abs(diff) > threshold).astype(np.float32)

        ramp_mask = mask.copy()
        ramp_mask[0] = 0
        ramp_mask[1:] *= mask[:-1]

        samples.append({
            "date":date,"site":site,
            "X":X,"y":y,"mask":mask,
            "base":base,"scale":scale,
            "residual":residual.astype(np.float32),
            "ramp_class":ramp_class,
            "ramp_binary":ramp_binary,
            "ramp_mask":ramp_mask,
        })
    return samples

train_samples = make_day_samples(train_ctx)
gap_samples = make_day_samples(gap_ctx)
temporal_samples = make_day_samples(temporal_ctx)

# Normalize using core-training samples only.
GEN_N = len(GEN_FEATURES)
MISS_START = GEN_N
WEATHER_START = GEN_N + len(MISSING_FEATURES)
WEATHER_END = WEATHER_START + len(WEATHER_FEATURES)

all_train_X = np.concatenate([s["X"] for s in train_samples],axis=0)
weather_mean = all_train_X[:,WEATHER_START:WEATHER_END].mean(axis=0)
weather_std = all_train_X[:,WEATHER_START:WEATHER_END].std(axis=0)+1e-6

def normalize_samples(samples):
    out = []
    for s in samples:
        z = copy.deepcopy(s)
        X = z["X"].copy()

        # Generation-like channels are scaled by each site's p99.
        X[:,:GEN_N] /= z["scale"]

        # Weather/interaction channels standardized from training only.
        X[:,WEATHER_START:WEATHER_END] = (
            X[:,WEATHER_START:WEATHER_END]-weather_mean
        )/weather_std

        z["X"] = np.nan_to_num(X,nan=0.0,posinf=0.0,neginf=0.0).astype(np.float32)
        out.append(z)
    return out

train_samples = normalize_samples(train_samples)
gap_samples = normalize_samples(gap_samples)
temporal_samples = normalize_samples(temporal_samples)

print("train site-days:",len(train_samples))
print("gap validation site-days:",len(gap_samples))
print("temporal validation site-days:",len(temporal_samples))
print("features per step:",train_samples[0]["X"].shape[1])


DEVICE: cpu
train site-days: 342
gap validation site-days: 36
temporal validation site-days: 63
features per step: 63


In [13]:

# ============================================================
# 11. Dataset
# ============================================================
class DayDataset(tud.Dataset):
    def __init__(self,samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,i):
        s = self.samples[i]
        return (
            torch.tensor(s["X"],dtype=torch.float32),
            torch.tensor(s["y"],dtype=torch.float32),
            torch.tensor(s["mask"],dtype=torch.float32),
            torch.tensor(s["base"],dtype=torch.float32),
            torch.tensor(s["scale"],dtype=torch.float32),
            torch.tensor(s["ramp_class"],dtype=torch.long),
            torch.tensor(s["ramp_binary"],dtype=torch.float32),
            torch.tensor(s["ramp_mask"],dtype=torch.float32),
        )

train_loader = tud.DataLoader(
    DayDataset(train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True
)



## 7. Architectures

All architectures share the same output heads so the comparison is fair.

- MLP
- LSTM
- BiLSTM
- GRU
- CNN-LSTM
- TCN
- Transformer Encoder

A true TFT is deliberately kept separate; a standard Transformer is **not mislabeled as TFT**.


In [14]:

# ============================================================
# 12. Deep-learning backbones
# ============================================================
class MLPBackbone(nn.Module):
    def __init__(self,n_features,hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features,hidden),nn.SiLU(),nn.Dropout(0.08),
            nn.Linear(hidden,hidden),nn.SiLU()
        )
        self.hidden = hidden
    def forward(self,x):
        return self.net(x)

class RNNBackbone(nn.Module):
    def __init__(self,n_features,kind="LSTM",hidden=64,bidirectional=False):
        super().__init__()
        h = hidden//2 if bidirectional else hidden
        cls = nn.LSTM if kind=="LSTM" else nn.GRU
        self.rnn = cls(
            n_features,h,batch_first=True,bidirectional=bidirectional
        )
        self.hidden = h*(2 if bidirectional else 1)
    def forward(self,x):
        z,_ = self.rnn(x)
        return z

class CNNLSTMBackbone(nn.Module):
    def __init__(self,n_features,hidden=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features,32,5,padding=2),
            nn.BatchNorm1d(32),nn.SiLU(),
            nn.Conv1d(32,32,3,padding=1),nn.SiLU()
        )
        self.rnn = nn.LSTM(32,hidden,batch_first=True)
        self.hidden = hidden
    def forward(self,x):
        z = self.conv(x.transpose(1,2)).transpose(1,2)
        z,_ = self.rnn(z)
        return z

class ResidualTCNBlock(nn.Module):
    def __init__(self,channels,dilation,dropout=0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels,channels,3,padding=dilation,dilation=dilation),
            nn.BatchNorm1d(channels),nn.SiLU(),nn.Dropout(dropout),
            nn.Conv1d(channels,channels,3,padding=dilation,dilation=dilation),
            nn.BatchNorm1d(channels),nn.SiLU()
        )
    def forward(self,x):
        return x+self.net(x)

class TCNBackbone(nn.Module):
    def __init__(self,n_features,hidden=128,dropout=0.1556):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv1d(n_features,hidden,1),nn.SiLU())
        self.blocks = nn.Sequential(
            *[ResidualTCNBlock(hidden,d,dropout) for d in [1,2,4,8]]
        )
        self.hidden = hidden
    def forward(self,x):
        z = self.proj(x.transpose(1,2))
        z = self.blocks(z)
        return z.transpose(1,2)

class TransformerBackbone(nn.Module):
    def __init__(self,n_features,hidden=64,nhead=4,layers=2):
        super().__init__()
        self.proj = nn.Linear(n_features,hidden)
        self.pos = nn.Parameter(torch.zeros(1,96,hidden))
        layer = nn.TransformerEncoderLayer(
            d_model=hidden,nhead=nhead,dim_feedforward=hidden*2,
            dropout=0.08,activation="gelu",batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer,num_layers=layers)
        self.hidden = hidden
    def forward(self,x):
        return self.encoder(self.proj(x)+self.pos[:,:x.shape[1]])

class ForecastMultiTaskModel(nn.Module):
    def __init__(self,backbone):
        super().__init__()
        self.backbone = backbone
        h = backbone.hidden
        self.reg_head = nn.Linear(h,1)
        self.class_head = nn.Linear(h,3)
        self.binary_head = nn.Linear(h,1)

    def forward(self,x):
        h = self.backbone(x)
        residual = self.reg_head(h).squeeze(-1)
        class_logits = self.class_head(h)
        binary_logit = self.binary_head(h).squeeze(-1)
        return residual,class_logits,binary_logit

def build_model(name,n_features,tcn_hidden=128,tcn_dropout=0.1556):
    if name=="MLP":
        bb = MLPBackbone(n_features)
    elif name=="LSTM":
        bb = RNNBackbone(n_features,"LSTM",64,False)
    elif name=="BiLSTM":
        bb = RNNBackbone(n_features,"LSTM",64,True)
    elif name=="GRU":
        bb = RNNBackbone(n_features,"GRU",64,False)
    elif name=="CNN-LSTM":
        bb = CNNLSTMBackbone(n_features)
    elif name=="TCN":
        bb = TCNBackbone(n_features,tcn_hidden,tcn_dropout)
    elif name=="Transformer":
        bb = TransformerBackbone(n_features)
    else:
        raise ValueError(name)
    return ForecastMultiTaskModel(bb)



## 8. Losses and optimizers

### Continuous MW regression
- MSE
- MAE
- Huber
- shifted MSLE — experimental, because the dataset can contain small negative MW measurements.
- Original composite = MSE + MAE + first-difference/ramp loss.

### Ramp classification
- BCE for large-ramp binary classification.
- categorical cross entropy for down/stable/up.
- hinge for a margin-based down/stable/up alternative.

### Optimizers
- SGD
- RMSProp
- AdamW
- Adagrad
- Adadelta


In [15]:

# ============================================================
# 13. Losses and optimizers
# ============================================================
TRAIN_MIN_MW = float(core_train["mw"].min())
MSLE_SHIFT = max(0.0,-TRAIN_MIN_MW)+0.10

def regression_loss(
    pred_mw,true_mw,mask,
    loss_name="Composite"
):
    valid = mask>0
    p = pred_mw[valid]
    y = true_mw[valid]

    mse = F.mse_loss(p,y)
    mae = F.l1_loss(p,y)

    if loss_name=="MSE":
        return mse
    if loss_name=="MAE":
        return mae
    if loss_name=="Huber":
        return F.smooth_l1_loss(p,y,beta=0.20)
    if loss_name=="MSLE":
        pp = torch.clamp(p+MSLE_SHIFT,min=0)
        yy = torch.clamp(y+MSLE_SHIFT,min=0)
        return F.mse_loss(torch.log1p(pp),torch.log1p(yy))
    if loss_name=="Composite":
        # First-difference shape loss only where both points are observed.
        dmask = mask[:,1:]*mask[:,:-1]
        dp = pred_mw[:,1:]-pred_mw[:,:-1]
        dy = true_mw[:,1:]-true_mw[:,:-1]
        ramp = (((dp-dy)**2)*dmask).sum()/dmask.sum().clamp_min(1)
        return 0.82*mse+0.13*mae+0.05*ramp

    raise ValueError(loss_name)

def categorical_hinge(logits,target,mask):
    valid = mask>0
    z = logits[valid]
    t = target[valid]
    true_score = z.gather(1,t[:,None]).squeeze(1)
    other = z.clone()
    other[torch.arange(len(t),device=z.device),t] = -1e9
    max_other = other.max(dim=1).values
    return torch.clamp(1-true_score+max_other,min=0).mean()

def auxiliary_loss(
    class_logits,binary_logit,
    ramp_class,ramp_binary,ramp_mask,
    class_mode="CCE"
):
    valid = ramp_mask>0

    # Binary cross entropy
    bce = F.binary_cross_entropy_with_logits(
        binary_logit[valid],ramp_binary[valid]
    )

    if class_mode=="CCE":
        multi = F.cross_entropy(
            class_logits[valid],ramp_class[valid]
        )
    elif class_mode=="HINGE":
        multi = categorical_hinge(
            class_logits,ramp_class,ramp_mask
        )
    else:
        raise ValueError(class_mode)

    return multi,bce

def make_optimizer(name,params,lr=None):
    default_lr = {
        "SGD":0.003,
        "RMSProp":0.0005,
        "AdamW":0.0010,
        "Adagrad":0.01,
        "Adadelta":1.0,
    }
    lr = default_lr[name] if lr is None else lr

    if name=="SGD":
        return torch.optim.SGD(params,lr=lr,momentum=0.9)
    if name=="RMSProp":
        return torch.optim.RMSprop(params,lr=lr)
    if name=="AdamW":
        return torch.optim.AdamW(params,lr=lr,weight_decay=1e-4)
    if name=="Adagrad":
        return torch.optim.Adagrad(params,lr=lr)
    if name=="Adadelta":
        return torch.optim.Adadelta(params,lr=lr)
    raise ValueError(name)


In [16]:

# ============================================================
# 14. Prediction and row reconstruction
# ============================================================
def predict_samples(model,samples):
    model.eval()
    row_map = {}
    ramp3_true,ramp3_pred = [],[]
    rampb_true,rampb_pred = [],[]

    with torch.no_grad():
        for s in samples:
            X = torch.tensor(s["X"],dtype=torch.float32,device=DEVICE)[None]
            res,cls_logits,bin_logit = model(X)

            res = res.cpu().numpy()[0]
            mw = s["base"]+res*s["scale"]

            cls = cls_logits.argmax(-1).cpu().numpy()[0]
            bcls = (torch.sigmoid(bin_logit)>=0.5).long().cpu().numpy()[0]

            for q,val in enumerate(mw):
                row_map[(s["date"],s["site"],q)] = float(val)

            rm = s["ramp_mask"]>0
            ramp3_true.extend(s["ramp_class"][rm].tolist())
            ramp3_pred.extend(cls[rm].tolist())
            rampb_true.extend(s["ramp_binary"][rm].astype(int).tolist())
            rampb_pred.extend(bcls[rm].tolist())

    cls_metrics = {
        "Ramp3":classification_metrics(ramp3_true,ramp3_pred),
        "RampBinary":classification_metrics(rampb_true,rampb_pred),
    }
    return row_map,cls_metrics

def map_to_rows(row_map,df):
    return np.array([
        row_map[(r["generation_date"],r["name"],int(r["q"]))]
        for _,r in df.iterrows()
    ],float)

def evaluate_model(model):
    gmap,gcls = predict_samples(model,gap_samples)
    tmap,tcls = predict_samples(model,temporal_samples)

    gp = physical_postprocess(map_to_rows(gmap,gap_ctx),gap_ctx,core_train)
    tp = physical_postprocess(map_to_rows(tmap,temporal_ctx),temporal_ctx,core_train)

    gm = regression_metrics(ygap,gp,naive=gap_ctx["profile_median"].fillna(0))
    tm = regression_metrics(ytemp,tp,naive=temporal_ctx["profile_median"].fillna(0))
    return gp,tp,gm,tm,gcls,tcls



## 9. Fifty-epoch trainer and the meaning of “best”

Every run executes **all 50 epochs**.

The checkpoint is selected by a normalized two-validation score:

\[
SelectionScore =
0.70\frac{GapRMSE}{GapBaselineRMSE}+
0.30\frac{TemporalRMSE}{TemporalBaselineRMSE}
\]

This prevents an easy validation split from dominating the decision.

The notebook reports:
- `BestEpoch`
- `Gap_RMSE`
- `Temporal_RMSE`
- `SelectionScore`

So **“best epoch”** and **“best model”** are no longer confused.


In [17]:
# ============================================================
# 15. Balanced model-selection score + 50-epoch trainer
# ============================================================
GAP_BASELINE_METRICS = regression_metrics(
    ygap,
    physical_postprocess(
        gap_ctx["profile_median"].fillna(0),
        gap_ctx,
        core_train
    )
)

TEMP_BASELINE_METRICS = regression_metrics(
    ytemp,
    physical_postprocess(
        temporal_ctx["profile_median"].fillna(0),
        temporal_ctx,
        core_train
    )
)

def _normalized_metric_score(metrics, baseline):
    return (
        RMSE_WEIGHT * (metrics["RMSE"] / baseline["RMSE"])
        + MAE_WEIGHT * (metrics["MAE"] / baseline["MAE"])
        + WMAPE_WEIGHT * (metrics["WMAPE_%"] / baseline["WMAPE_%"])
    )

def selection_score(gap_metrics, temporal_metrics):
    """Lower is better."""
    gap_part = _normalized_metric_score(
        gap_metrics,
        GAP_BASELINE_METRICS
    )
    temporal_part = _normalized_metric_score(
        temporal_metrics,
        TEMP_BASELINE_METRICS
    )
    return (
        GAP_WEIGHT * gap_part
        + TEMPORAL_WEIGHT * temporal_part
    )

def train_model_50(
    architecture,
    regression_loss_name="Composite",
    optimizer_name="AdamW",
    aux_mode="NONE",
    seed=SEED,
    lr=None,
    tcn_hidden=128,
    tcn_dropout=0.1556,
    verbose=False,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    n_features = train_samples[0]["X"].shape[1]
    model = build_model(
        architecture,
        n_features,
        tcn_hidden=tcn_hidden,
        tcn_dropout=tcn_dropout
    ).to(DEVICE)

    optimizer = make_optimizer(optimizer_name, model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=6,
        min_lr=1e-5
    )

    best = {
        "score": np.inf,
        "epoch": None,
        "state": None,
        "gap": None,
        "temporal": None,
    }
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        batch_losses = []

        for X,y,mask,base,scale,ramp_class,ramp_binary,ramp_mask in train_loader:
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)
            base = base.to(DEVICE)
            scale = scale.to(DEVICE)
            ramp_class = ramp_class.to(DEVICE)
            ramp_binary = ramp_binary.to(DEVICE)
            ramp_mask = ramp_mask.to(DEVICE)

            optimizer.zero_grad()

            residual_pred, class_logits, binary_logit = model(X)
            pred_mw = base + residual_pred * scale[:, None]

            reg = regression_loss(
                pred_mw,
                y,
                mask,
                regression_loss_name
            )

            total = reg

            if aux_mode != "NONE":
                class_mode = "CCE" if aux_mode == "CCE_BCE" else "HINGE"
                multi, bce = auxiliary_loss(
                    class_logits,
                    binary_logit,
                    ramp_class,
                    ramp_binary,
                    ramp_mask,
                    class_mode=class_mode
                )
                total = reg + 0.02 * multi + 0.02 * bce

            if not torch.isfinite(total):
                raise RuntimeError(
                    f"Non-finite loss: {architecture}, "
                    f"{regression_loss_name}, {optimizer_name}"
                )

            total.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
            batch_losses.append(float(total.detach().cpu()))

        gp,tp,gm,tm,_,_ = evaluate_model(model)
        sel = selection_score(gm, tm)
        scheduler.step(sel)

        history.append({
            "Epoch": epoch,
            "TrainLoss": float(np.mean(batch_losses)),
            "Gap_RMSE": gm["RMSE"],
            "Gap_MAE": gm["MAE"],
            "Gap_WMAPE_%": gm["WMAPE_%"],
            "Temporal_RMSE": tm["RMSE"],
            "Temporal_MAE": tm["MAE"],
            "Temporal_WMAPE_%": tm["WMAPE_%"],
            "SelectionScore": sel,
        })

        if sel < best["score"]:
            best = {
                "score": sel,
                "epoch": epoch,
                "state": copy.deepcopy(model.state_dict()),
                "gap": gm,
                "temporal": tm,
            }

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(
                architecture,
                epoch,
                "gap RMSE", round(gm["RMSE"], 5),
                "temp RMSE", round(tm["RMSE"], 5),
                "balanced score", round(sel, 5)
            )

    model.load_state_dict(best["state"])
    return model, pd.DataFrame(history), best



## 10. Architecture benchmark

Same:
- core training data,
- gap validation,
- temporal validation,
- 50 epochs,
- loss,
- optimizer,
- random seed.

Only the architecture changes.


In [18]:

# ============================================================
# 16. Architecture benchmark
# ============================================================
DEEP_ARCHITECTURES = [
    "MLP","LSTM","BiLSTM","GRU","CNN-LSTM","TCN","Transformer"
]

deep_models = {}
deep_predictions = {"gap":{}, "temporal":{}}
deep_rows = []
deep_histories = {}

if RUN_ARCHITECTURES:
    for arch in DEEP_ARCHITECTURES:
        print("Training architecture:",arch)

        # Use the Optuna-discovered TCN neighborhood from the previous Colab
        # as the starting TCN configuration, but still compare fairly.
        lr = 0.0018974773610953666 if arch=="TCN" else None

        model,hist,best = train_model_50(
            arch,
            regression_loss_name="Composite",
            optimizer_name="AdamW",
            aux_mode="NONE",
            seed=SEED,
            lr=lr,
            tcn_hidden=128,
            tcn_dropout=0.15562200257933947,
        )

        gp,tp,gm,tm,gcls,tcls = evaluate_model(model)

        deep_models[arch] = model
        deep_histories[arch] = hist
        deep_predictions["gap"][arch] = gp
        deep_predictions["temporal"][arch] = tp
        prediction_pool["gap"][arch] = gp
        prediction_pool["temporal"][arch] = tp

        row = {
            "Model":arch,
            "BestEpoch":best["epoch"],
            "SelectionScore":selection_score(gm,tm),
            **{f"Gap_{k}":v for k,v in gm.items()},
            **{f"Temporal_{k}":v for k,v in tm.items()},
        }
        deep_rows.append(row)

    deep_table = pd.DataFrame(deep_rows).set_index("Model").sort_values("SelectionScore")
    display(deep_table[
        ["BestEpoch","SelectionScore","Gap_RMSE","Gap_MAE","Gap_WMAPE_%",
         "Temporal_RMSE","Temporal_MAE","Temporal_WMAPE_%"]
    ].round(4))
else:
    deep_table = pd.DataFrame()


Training architecture: MLP
Training architecture: LSTM
Training architecture: BiLSTM
Training architecture: GRU
Training architecture: CNN-LSTM
Training architecture: TCN
Training architecture: Transformer


,BestEpoch,SelectionScore,Gap_RMSE,Gap_MAE,Gap_WMAPE_%,Temporal_RMSE,Temporal_MAE,Temporal_WMAPE_%
Model,,,,,,,,
MLP,6,0.9400,0.9200,0.4561,38.0374,0.8734,0.4347,40.9095
GRU,5,0.9449,0.9224,0.4512,37.6260,0.8974,0.4457,41.9491
CNN-LSTM,3,0.9482,0.9322,0.4590,38.2809,0.8895,0.4327,40.7215
BiLSTM,5,0.9482,0.9360,0.4651,38.7836,0.8587,0.4305,40.5207
Transformer,4,0.9575,0.9208,0.4725,39.4023,0.8798,0.4474,42.1139
LSTM,3,0.9618,0.9397,0.4760,39.6953,0.8708,0.4376,41.1866
TCN,21,0.9689,0.9567,0.4762,39.7151,0.8765,0.4379,41.2195



## 11. Loss-function benchmark

The best architecture is now held fixed.

We compare:
- MSE
- MAE
- Huber
- shifted MSLE
- original Composite

All for 50 epochs.


In [19]:

# ============================================================
# 17. Loss benchmark
# ============================================================
if RUN_LOSS_BENCHMARK and len(deep_table):
    best_arch = deep_table.index[0]
    loss_rows = []
    loss_models = {}

    for loss_name in ["MSE","MAE","Huber","MSLE","Composite"]:
        print("Loss:",loss_name)

        lr = 0.0018974773610953666 if best_arch=="TCN" else None

        model,hist,best = train_model_50(
            best_arch,
            regression_loss_name=loss_name,
            optimizer_name="AdamW",
            aux_mode="NONE",
            seed=SEED,
            lr=lr,
            tcn_hidden=128,
            tcn_dropout=0.15562200257933947,
        )

        gp,tp,gm,tm,_,_ = evaluate_model(model)
        loss_models[loss_name] = model

        loss_rows.append({
            "Loss":loss_name,
            "BestEpoch":best["epoch"],
            "SelectionScore":selection_score(gm,tm),
            **{f"Gap_{k}":v for k,v in gm.items()},
            **{f"Temporal_{k}":v for k,v in tm.items()},
        })

    loss_table = pd.DataFrame(loss_rows).set_index("Loss").sort_values("SelectionScore")
    display(loss_table[
        ["BestEpoch","SelectionScore","Gap_RMSE","Gap_MAE","Gap_WMAPE_%",
         "Temporal_RMSE","Temporal_MAE","Temporal_WMAPE_%"]
    ].round(4))
else:
    loss_table = pd.DataFrame()
    best_arch = deep_table.index[0] if len(deep_table) else "TCN"


Loss: MSE
Loss: MAE
Loss: Huber
Loss: MSLE
Loss: Composite


,BestEpoch,SelectionScore,Gap_RMSE,Gap_MAE,Gap_WMAPE_%,Temporal_RMSE,Temporal_MAE,Temporal_WMAPE_%
Loss,,,,,,,,
MSE,6,0.9393,0.9186,0.4562,38.0426,0.8722,0.4347,40.9122
Composite,6,0.9400,0.9200,0.4561,38.0374,0.8734,0.4347,40.9095
Huber,17,0.9656,0.9521,0.4614,38.4759,0.9170,0.4454,41.9221
MAE,17,0.9688,0.9587,0.4601,38.3691,0.9263,0.4460,41.9809
MSLE,3,0.9727,0.9622,0.4710,39.2766,0.9058,0.4406,41.4663



## 12. Optimizer benchmark

The best architecture and regression loss are fixed.

Compare:
**SGD → RMSProp → AdamW → Adagrad → Adadelta**

All for 50 epochs.


In [20]:

# ============================================================
# 18. Optimizer benchmark
# ============================================================
if RUN_OPTIMIZER_BENCHMARK and len(loss_table):
    best_loss = loss_table.index[0]
    optimizer_rows = []
    optimizer_models = {}

    for opt_name in ["SGD","RMSProp","AdamW","Adagrad","Adadelta"]:
        print("Optimizer:",opt_name)

        model,hist,best = train_model_50(
            best_arch,
            regression_loss_name=best_loss,
            optimizer_name=opt_name,
            aux_mode="NONE",
            seed=SEED,
            tcn_hidden=128,
            tcn_dropout=0.15562200257933947,
        )

        gp,tp,gm,tm,_,_ = evaluate_model(model)
        optimizer_models[opt_name] = model

        optimizer_rows.append({
            "Optimizer":opt_name,
            "BestEpoch":best["epoch"],
            "SelectionScore":selection_score(gm,tm),
            **{f"Gap_{k}":v for k,v in gm.items()},
            **{f"Temporal_{k}":v for k,v in tm.items()},
        })

    optimizer_table = pd.DataFrame(optimizer_rows).set_index("Optimizer").sort_values("SelectionScore")
    display(optimizer_table[
        ["BestEpoch","SelectionScore","Gap_RMSE","Gap_MAE","Gap_WMAPE_%",
         "Temporal_RMSE","Temporal_MAE","Temporal_WMAPE_%"]
    ].round(4))
else:
    optimizer_table = pd.DataFrame()
    best_loss = loss_table.index[0] if len(loss_table) else "Composite"


Optimizer: SGD
Optimizer: RMSProp
Optimizer: AdamW
Optimizer: Adagrad
Optimizer: Adadelta


,BestEpoch,SelectionScore,Gap_RMSE,Gap_MAE,Gap_WMAPE_%,Temporal_RMSE,Temporal_MAE,Temporal_WMAPE_%
Optimizer,,,,,,,,
Adadelta,7,0.9342,0.9068,0.4487,37.4161,0.8838,0.4421,41.6150
SGD,14,0.9348,0.9123,0.4539,37.8519,0.8687,0.4345,40.8926
RMSProp,5,0.9389,0.9204,0.4546,37.9126,0.8740,0.4339,40.8370
AdamW,6,0.9393,0.9186,0.4562,38.0426,0.8722,0.4347,40.9122
Adagrad,5,0.9401,0.9211,0.4548,37.9285,0.8762,0.4353,40.9713



## 13. BCE / CCE / hinge and classification metrics

This experiment asks whether a small auxiliary grid-ramp objective improves the shared representation.

Configurations:
- `NONE`
- `CCE_BCE`
- `HINGE_BCE`

Regression is still judged by RMSE/MAE/WMAPE.

Ramp-event classification is judged by:
- Accuracy
- Precision
- Recall
- F1


In [21]:

# ============================================================
# 19. Auxiliary classification benchmark
# ============================================================
if RUN_AUXILIARY_BENCHMARK and len(optimizer_table):
    best_optimizer = optimizer_table.index[0]
    aux_rows = []
    aux_models = {}

    for aux_mode in ["NONE","CCE_BCE","HINGE_BCE"]:
        print("Auxiliary mode:",aux_mode)

        model,hist,best = train_model_50(
            best_arch,
            regression_loss_name=best_loss,
            optimizer_name=best_optimizer,
            aux_mode=aux_mode,
            seed=SEED,
            tcn_hidden=128,
            tcn_dropout=0.15562200257933947,
        )

        gp,tp,gm,tm,gcls,tcls = evaluate_model(model)
        aux_models[aux_mode] = model

        aux_rows.append({
            "AuxMode":aux_mode,
            "BestEpoch":best["epoch"],
            "SelectionScore":selection_score(gm,tm),
            "Gap_RMSE":gm["RMSE"],
            "Temporal_RMSE":tm["RMSE"],
            "Gap_Ramp3_Accuracy":gcls["Ramp3"]["Accuracy"],
            "Gap_Ramp3_Recall":gcls["Ramp3"]["Recall_macro"],
            "Gap_Ramp3_F1":gcls["Ramp3"]["F1_macro"],
            "Gap_RampBinary_Accuracy":gcls["RampBinary"]["Accuracy"],
            "Gap_RampBinary_Recall":gcls["RampBinary"]["Recall_macro"],
            "Gap_RampBinary_F1":gcls["RampBinary"]["F1_macro"],
        })

    auxiliary_table = pd.DataFrame(aux_rows).set_index("AuxMode").sort_values("SelectionScore")
    display(auxiliary_table.round(4))
else:
    auxiliary_table = pd.DataFrame()
    best_optimizer = optimizer_table.index[0] if len(optimizer_table) else "AdamW"


Auxiliary mode: NONE
Auxiliary mode: CCE_BCE
Auxiliary mode: HINGE_BCE


,BestEpoch,SelectionScore,Gap_RMSE,Temporal_RMSE,Gap_Ramp3_Accuracy,Gap_Ramp3_Recall,Gap_Ramp3_F1,Gap_RampBinary_Accuracy,Gap_RampBinary_Recall,Gap_RampBinary_F1
AuxMode,,,,,,,,,,
NONE,7,0.9342,0.9068,0.8838,0.6770,0.3494,0.3179,0.3162,0.5038,0.2574
CCE_BCE,7,0.9343,0.9067,0.8840,0.6956,0.3332,0.2735,0.6953,0.5009,0.4148
HINGE_BCE,7,0.9343,0.9066,0.8840,0.6959,0.3333,0.2736,0.6959,0.5019,0.4168



## 14. Unified scoreboard and Pareto front

A deep model is only a winner if it beats the baseline/ML alternatives.

The Pareto front identifies models for which no other model is simultaneously better in:
- RMSE
- MAE
- WMAPE


In [22]:
# ============================================================
# 20. Unified scoreboard + Pareto
# ============================================================
score_rows = []

family_lookup = {}
if len(classical_table):
    family_lookup.update(classical_table["Family"].to_dict())
for arch in DEEP_ARCHITECTURES:
    family_lookup[arch] = "Deep Learning"

for name, gp in prediction_pool["gap"].items():
    if name not in prediction_pool["temporal"]:
        continue

    tp = prediction_pool["temporal"][name]
    gm = regression_metrics(ygap, gp)
    tm = regression_metrics(ytemp, tp)

    score_rows.append({
        "Model": name,
        "Family": family_lookup.get(name, "Other"),
        "SelectionScore": selection_score(gm, tm),
        **{f"Gap_{k}": v for k, v in gm.items()},
        **{f"Temporal_{k}": v for k, v in tm.items()},
    })

scoreboard = (
    pd.DataFrame(score_rows)
    .drop_duplicates("Model")
    .set_index("Model")
    .sort_values(["SelectionScore","Gap_RMSE","Gap_MAE"])
)

print("UNIFIED FORECASTING SCOREBOARD — lower SelectionScore is better")
display(
    scoreboard[
        [
            "Family","SelectionScore",
            "Gap_RMSE","Gap_MAE","Gap_WMAPE_%",
            "Temporal_RMSE","Temporal_MAE","Temporal_WMAPE_%"
        ]
    ].round(4)
)

def pareto_front(df, metrics):
    names = list(df.index)
    keep = []

    for i, a in enumerate(names):
        va = df.loc[a, metrics].to_numpy(float)
        dominated = False

        for j, b in enumerate(names):
            if i == j:
                continue

            vb = df.loc[b, metrics].to_numpy(float)
            if np.all(vb <= va) and np.any(vb < va):
                dominated = True
                break

        if not dominated:
            keep.append(a)

    return df.loc[keep]

gap_pareto = pareto_front(
    scoreboard,
    ["Gap_RMSE","Gap_MAE","Gap_WMAPE_%"]
).sort_values("SelectionScore")

print("Gap-validation Pareto front:")
display(
    gap_pareto[
        ["Family","SelectionScore","Gap_RMSE","Gap_MAE","Gap_WMAPE_%"]
    ].round(4)
)


UNIFIED FORECASTING SCOREBOARD — lower SelectionScore is better


,Family,SelectionScore,Gap_RMSE,Gap_MAE,Gap_WMAPE_%,Temporal_RMSE,Temporal_MAE,Temporal_WMAPE_%
Model,,,,,,,,
XGBoost,Machine Learning,0.9325,0.9005,0.4553,37.9667,0.8740,0.4341,40.8569
Ridge,Classical,0.9333,0.9096,0.4605,38.4051,0.8396,0.4324,40.7022
LightGBM,Machine Learning,0.9390,0.9060,0.4543,37.8877,0.8886,0.4426,41.6618
MLP,Deep Learning,0.9400,0.9200,0.4561,38.0374,0.8734,0.4347,40.9095
GRU,Deep Learning,0.9449,0.9224,0.4512,37.6260,0.8974,0.4457,41.9491
CNN-LSTM,Deep Learning,0.9482,0.9322,0.4590,38.2809,0.8895,0.4327,40.7215
BiLSTM,Deep Learning,0.9482,0.9360,0.4651,38.7836,0.8587,0.4305,40.5207
RandomForest,Machine Learning,0.9497,0.8901,0.4490,37.4460,0.9632,0.4662,43.8755
Transformer,Deep Learning,0.9575,0.9208,0.4725,39.4023,0.8798,0.4474,42.1139


Gap-validation Pareto front:


,Family,SelectionScore,Gap_RMSE,Gap_MAE,Gap_WMAPE_%
Model,,,,,
RandomForest,Machine Learning,0.9497,0.8901,0.449,37.446



## 15. Objective-specific ensemble weights

The previous notebook used a fixed 44/56 hybrid. In this version the weight is **measured**, not assumed.

For the top three candidates, the notebook searches nonnegative weights that sum to 1 and produces separate winners for:

- minimum RMSE,
- minimum MAE,
- minimum WMAPE,
- balanced normalized RMSE+MAE+WMAPE.

This directly answers the question: **“Best according to which measurement?”**


In [23]:
# ============================================================
# 21. Validation-tuned ensemble search
# ============================================================
top_models = list(scoreboard.index[:3])
print("Top ensemble candidates:", top_models)

def objective_value(metric, gap_pred, temp_pred):
    gm = regression_metrics(ygap, gap_pred)
    tm = regression_metrics(ytemp, temp_pred)

    if metric == "CONTEST":
        return selection_score(gm, tm)

    if metric == "RMSE":
        return (
            GAP_WEIGHT * (gm["RMSE"] / GAP_BASELINE_METRICS["RMSE"])
            + TEMPORAL_WEIGHT * (tm["RMSE"] / TEMP_BASELINE_METRICS["RMSE"])
        )

    if metric == "MAE":
        return (
            GAP_WEIGHT * (gm["MAE"] / GAP_BASELINE_METRICS["MAE"])
            + TEMPORAL_WEIGHT * (tm["MAE"] / TEMP_BASELINE_METRICS["MAE"])
        )

    if metric == "WMAPE":
        return (
            GAP_WEIGHT * (gm["WMAPE_%"] / GAP_BASELINE_METRICS["WMAPE_%"])
            + TEMPORAL_WEIGHT * (tm["WMAPE_%"] / TEMP_BASELINE_METRICS["WMAPE_%"])
        )

    raise ValueError(metric)

def search_ensemble(metric):
    best = {
        "value": np.inf,
        "weights": None,
        "gap_pred": None,
        "temp_pred": None,
    }

    grid = np.arange(0, 1.0001, 0.05)

    for w1 in grid:
        for w2 in grid:
            w3 = 1.0 - w1 - w2
            if w3 < -1e-9 or w3 > 1:
                continue

            ws = [w1, w2, w3]

            gp = sum(
                w * prediction_pool["gap"][m]
                for w, m in zip(ws, top_models)
            )
            tp = sum(
                w * prediction_pool["temporal"][m]
                for w, m in zip(ws, top_models)
            )

            gp = physical_postprocess(gp, gap_ctx, core_train)
            tp = physical_postprocess(tp, temporal_ctx, core_train)

            value = objective_value(metric, gp, tp)

            if value < best["value"]:
                best = {
                    "value": value,
                    "weights": ws,
                    "gap_pred": gp.copy(),
                    "temp_pred": tp.copy(),
                }

    return best

ensemble_results = {}
ens_rows = []

for objective in ["CONTEST","RMSE","MAE","WMAPE"]:
    result = search_ensemble(objective)
    ensemble_results[objective] = result

    gm = regression_metrics(ygap, result["gap_pred"])
    tm = regression_metrics(ytemp, result["temp_pred"])

    ens_rows.append({
        "Objective": objective,
        "SelectionScore": selection_score(gm, tm),
        "Weights": dict(zip(top_models, result["weights"])),
        "Gap_RMSE": gm["RMSE"],
        "Gap_MAE": gm["MAE"],
        "Gap_WMAPE_%": gm["WMAPE_%"],
        "Temporal_RMSE": tm["RMSE"],
        "Temporal_MAE": tm["MAE"],
        "Temporal_WMAPE_%": tm["WMAPE_%"],
    })

ensemble_table = (
    pd.DataFrame(ens_rows)
    .set_index("Objective")
    .sort_values("SelectionScore")
)

display(ensemble_table.round(4))


Top ensemble candidates: ['XGBoost', 'Ridge', 'LightGBM']


,SelectionScore,Weights,Gap_RMSE,Gap_MAE,Gap_WMAPE_%,Temporal_RMSE,Temporal_MAE,Temporal_WMAPE_%
Objective,,,,,,,,
CONTEST,0.9181,"{'XGBoost': 0.45, 'Ridge': 0.45, 'LightGBM': 0...",0.8931,0.4517,37.6688,0.8407,0.4230,39.8136
MAE,0.9184,"{'XGBoost': 0.45, 'Ridge': 0.4, 'LightGBM': 0....",0.8927,0.4513,37.6406,0.8433,0.4235,39.8619
WMAPE,0.9184,"{'XGBoost': 0.45, 'Ridge': 0.4, 'LightGBM': 0....",0.8927,0.4513,37.6406,0.8433,0.4235,39.8619
RMSE,0.9185,"{'XGBoost': 0.4, 'Ridge': 0.55, 'LightGBM': 0....",0.8943,0.4526,37.7425,0.8370,0.4228,39.7939



## 16. Confidence intervals

A score difference of only a few thousandths of MW may be noise.

The bootstrap resamples **whole days**, preserving the time-series dependence within each day, and returns a 95% confidence interval.


In [24]:

# ============================================================
# 22. Day-block bootstrap confidence interval
# ============================================================
def day_block_bootstrap(df,pred,n_boot=1000,seed=SEED):
    rng = np.random.default_rng(seed)

    z = df[["generation_date","mw"]].copy()
    z["pred"] = np.asarray(pred)
    days = z["generation_date"].unique()

    vals = []
    for _ in range(n_boot):
        sampled_days = rng.choice(days,size=len(days),replace=True)
        sample = pd.concat(
            [z[z["generation_date"]==d] for d in sampled_days],
            ignore_index=True
        )
        m = regression_metrics(sample["mw"],sample["pred"])
        vals.append([m["RMSE"],m["MAE"],m["WMAPE_%"]])

    a = np.asarray(vals)
    return pd.DataFrame({
        "Metric":["RMSE","MAE","WMAPE_%"],
        "Lower95":np.quantile(a,0.025,axis=0),
        "Median":np.quantile(a,0.50,axis=0),
        "Upper95":np.quantile(a,0.975,axis=0),
    })

winner_name = scoreboard.index[0]
print("Best individual validation model:",winner_name)
display(day_block_bootstrap(
    gap_ctx,prediction_pool["gap"][winner_name]
).round(4))

print("Best RMSE ensemble:")
display(day_block_bootstrap(
    gap_ctx,ensemble_results["RMSE"]["gap_pred"]
).round(4))


Best individual validation model: XGBoost


,Metric,Lower95,Median,Upper95
0,RMSE,0.8461,0.9005,0.9557
1,MAE,0.4459,0.4553,0.4664
2,WMAPE_%,35.7705,37.9667,41.0850


Best RMSE ensemble:


,Metric,Lower95,Median,Upper95
0,RMSE,0.8222,0.8943,0.9632
1,MAE,0.4359,0.4526,0.4690
2,WMAPE_%,35.1936,37.7425,41.3112



## 17. Optional: apply hyperparameter tuning correctly

The previous notebook found a better Optuna trial but did not retrain the final global model with the winning parameters.

The correct workflow is:

**Tune → record best parameters → rebuild model → train 50 epochs → restore best checkpoint → compare on untouched validation.**

The full architecture/loss/optimizer benchmark above usually provides a stronger contest story than optimizing only TCN.


In [25]:

# ============================================================
# 23. Optional Optuna refinement of the selected architecture
# ============================================================
# RUN_OPTUNA is configured in Cell 0.
if RUN_OPTUNA:
    import subprocess,sys
    try:
        import optuna
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","optuna"])
        import optuna

    def objective(trial):
        hidden = trial.suggest_categorical("tcn_hidden",[64,96,128,160])
        dropout = trial.suggest_float("tcn_dropout",0.03,0.22)
        lr = trial.suggest_float("lr",2e-4,3e-3,log=True)

        # Tuning phase is intentionally shorter.
        # The winning configuration MUST be refit for the requested full 50 epochs.
        old_epochs = globals()["EPOCHS"]
        globals()["EPOCHS"] = 20

        try:
            model,hist,best = train_model_50(
                best_arch,
                regression_loss_name=best_loss,
                optimizer_name=best_optimizer,
                aux_mode="NONE",
                seed=SEED,
                lr=lr,
                tcn_hidden=hidden,
                tcn_dropout=dropout
            )
            value = best["score"]
        finally:
            globals()["EPOCHS"] = old_epochs

        return value

    study = optuna.create_study(direction="minimize")
    study.optimize(objective,n_trials=12)

    print("Optuna best:",study.best_params,study.best_value)

    # CRITICAL: rebuild + refit the tuned winner for the full requested 50 epochs.
    tuned_final_model,tuned_hist,tuned_best = train_model_50(
        best_arch,
        regression_loss_name=best_loss,
        optimizer_name=best_optimizer,
        aux_mode="NONE",
        seed=SEED,
        lr=study.best_params["lr"],
        tcn_hidden=study.best_params.get("tcn_hidden",128),
        tcn_dropout=study.best_params.get("tcn_dropout",0.10)
    )

    print("Refit tuned best epoch:",tuned_best["epoch"])
    print("Refit tuned selection score:",tuned_best["score"])



## 18. Multi-seed deep ensemble

Deep networks are stochastic. A single seed can make a configuration look unusually good or bad.

For the final selected deep configuration:
- train seeds 230, 42 and 2026,
- each for 50 epochs,
- restore each seed's best checkpoint,
- average predictions.

This is often more robust than trusting one lucky run.


In [26]:

# ============================================================
# 24. Multi-seed final selected deep configuration
# ============================================================
seed_models = []
seed_gap_predictions = []
seed_temp_predictions = []
seed_rows = []

# Prefer choices from completed stages.
selected_arch = best_arch
selected_loss = best_loss
selected_optimizer = best_optimizer
selected_aux = (
    auxiliary_table.index[0]
    if len(auxiliary_table) else "NONE"
)

if RUN_MULTI_SEED_FINAL:
    for sd in MULTI_SEEDS:
        print("Final seed:",sd)

        model,hist,best = train_model_50(
            selected_arch,
            regression_loss_name=selected_loss,
            optimizer_name=selected_optimizer,
            aux_mode=selected_aux,
            seed=sd,
            tcn_hidden=128,
            tcn_dropout=0.15562200257933947
        )

        gp,tp,gm,tm,gcls,tcls = evaluate_model(model)

        seed_models.append(model)
        seed_gap_predictions.append(gp)
        seed_temp_predictions.append(tp)

        seed_rows.append({
            "Seed":sd,
            "BestEpoch":best["epoch"],
            "SelectionScore":selection_score(gm,tm),
            "Gap_RMSE":gm["RMSE"],
            "Gap_MAE":gm["MAE"],
            "Temporal_RMSE":tm["RMSE"],
            "Temporal_MAE":tm["MAE"],
        })

    seed_table = pd.DataFrame(seed_rows).set_index("Seed")
    display(seed_table.round(4))

    seed_ensemble_gap = np.mean(seed_gap_predictions,axis=0)
    seed_ensemble_temp = np.mean(seed_temp_predictions,axis=0)

    seed_ensemble_gap = physical_postprocess(seed_ensemble_gap,gap_ctx,core_train)
    seed_ensemble_temp = physical_postprocess(seed_ensemble_temp,temporal_ctx,core_train)

    print("Multi-seed gap metrics:")
    print(regression_metrics(ygap,seed_ensemble_gap))
    print("Multi-seed temporal metrics:")
    print(regression_metrics(ytemp,seed_ensemble_temp))


Final seed: 230
Final seed: 42
Final seed: 2026


,BestEpoch,SelectionScore,Gap_RMSE,Gap_MAE,Temporal_RMSE,Temporal_MAE
Seed,,,,,,
230,7,0.9342,0.9068,0.4487,0.8838,0.4421
42,4,0.9419,0.9175,0.4480,0.9020,0.4466
2026,2,0.9457,0.9198,0.4483,0.9030,0.4541


Multi-seed gap metrics:
{'RMSE': 0.9101768256722585, 'MAE': 0.4461511766022933, 'MAPE_eps_%': 59.61172102210001, 'Active_MAPE_%': 66.67306683846887, 'WMAPE_%': 37.20724060983475, 'sMAPE_%': 31.066417689292013, 'R2': 0.7628361291943087, 'MASE': nan}
Multi-seed temporal metrics:
{'RMSE': 0.8929804007714967, 'MAE': 0.4455185194252882, 'MAPE_eps_%': 91.58445118041044, 'Active_MAPE_%': 96.98788659821709, 'WMAPE_%': 41.932321618804295, 'sMAPE_%': 34.59434682596797, 'R2': 0.726420598139933, 'MASE': nan}



## 18.1 Final contest blend: Deep + Seasonal Profile + Bidirectional Neighbor

The previous notebook used a fixed 44/56 blend.

Here the final lightweight blend is selected **from validation**:

- selected deep model / multi-seed deep ensemble,
- seasonal profile median,
- bidirectional ±1-day neighbor estimate.

Weights are nonnegative, sum to 1, and are searched in 0.05 increments.

This blend is easy to reproduce on the real test set because all three components can be generated without using test labels.


In [27]:

# ============================================================
# 24B. Validation-tuned final lightweight blend
# ============================================================
if RUN_MULTI_SEED_FINAL:
    deep_gap_for_final = seed_ensemble_gap
    deep_temp_for_final = seed_ensemble_temp
else:
    # Fallback to the selected architecture's stored benchmark prediction.
    deep_gap_for_final = prediction_pool["gap"].get(
        selected_arch,
        prediction_pool["gap"][scoreboard.index[0]]
    )
    deep_temp_for_final = prediction_pool["temporal"].get(
        selected_arch,
        prediction_pool["temporal"][scoreboard.index[0]]
    )

profile_gap = prediction_pool["gap"]["Seasonal_Profile_Median"]
profile_temp = prediction_pool["temporal"]["Seasonal_Profile_Median"]

neighbor_gap = gap_ctx[["prev_1d","next_1d"]].mean(axis=1)
neighbor_gap = neighbor_gap.fillna(gap_ctx["profile_median"]).to_numpy()

# Temporal validation intentionally has no future-context values.
neighbor_temp = temporal_ctx["prev_1d"].fillna(
    temporal_ctx["profile_median"]
).to_numpy()

best_final_blend = {
    "score":np.inf,
    "weights":None,
    "gap_pred":None,
    "temp_pred":None,
}

grid = np.arange(0,1.0001,0.05)

for wd in grid:
    for wp in grid:
        wn = 1.0-wd-wp
        if wn < -1e-9 or wn > 1:
            continue

        gp = wd*deep_gap_for_final + wp*profile_gap + wn*neighbor_gap
        tp = wd*deep_temp_for_final + wp*profile_temp + wn*neighbor_temp

        gp = physical_postprocess(gp,gap_ctx,core_train)
        tp = physical_postprocess(tp,temporal_ctx,core_train)

        gm = regression_metrics(ygap,gp)
        tm = regression_metrics(ytemp,tp)

        # Contest-first, but still penalize a model that fails badly on temporal robustness.
        score_v = selection_score(gm,tm)

        if score_v < best_final_blend["score"]:
            best_final_blend = {
                "score":score_v,
                "weights":{"Deep":wd,"Profile":wp,"Neighbor":wn},
                "gap_pred":gp.copy(),
                "temp_pred":tp.copy(),
            }

FINAL_COMPONENT_WEIGHTS = best_final_blend["weights"]

print("Final validation-tuned component weights:",FINAL_COMPONENT_WEIGHTS)
print("Gap metrics:",regression_metrics(ygap,best_final_blend["gap_pred"]))
print("Temporal metrics:",regression_metrics(ytemp,best_final_blend["temp_pred"]))


Final validation-tuned component weights: {'Deep': np.float64(0.9), 'Profile': np.float64(0.0), 'Neighbor': np.float64(0.09999999999999998)}
Gap metrics: {'RMSE': 0.9077187702275337, 'MAE': 0.44443764800125307, 'MAPE_eps_%': 58.02126240232942, 'Active_MAPE_%': 65.13258079676753, 'WMAPE_%': 37.064339113000706, 'sMAPE_%': 30.945602439848557, 'R2': 0.7641153855844375, 'MASE': nan}
Temporal metrics: {'RMSE': 0.8957924200966895, 'MAE': 0.44508877226516197, 'MAPE_eps_%': 89.57362891670961, 'Active_MAPE_%': 94.77373709849665, 'WMAPE_%': 41.891873701720115, 'sMAPE_%': 34.652922362696735, 'R2': 0.7246948674202423, 'MASE': nan}


## 18.2 🏆 Automatic Champion Selection — Best Forecasting Model

The notebook compares every final candidate strategy with the **same balanced validation score**:

- Best individual model from the unified scoreboard
- Best validation-tuned Top-3 Ensemble
- Tuned Multi-Seed Deep model
- Deep + Seasonal Profile + Neighbor blend

### Selection rule

**Lower is better**

- Gap / Contest-Mimic Validation = 70%
- Temporal / Future Validation = 30%

Inside each validation view:

- RMSE = 50%
- MAE = 30%
- WMAPE = 20%

The final test prediction must follow the selected champion. It is no longer forced back to a Deep Learning architecture when another strategy validates better.


In [28]:
# ============================================================
# 24C. AUTOMATIC CHAMPION SELECTION
# ============================================================
from IPython.display import display, HTML

champion_candidates = {}

# A) Best individual model
best_individual_name = scoreboard.index[0]
best_individual_gap = prediction_pool["gap"][best_individual_name]
best_individual_temp = prediction_pool["temporal"][best_individual_name]

best_individual_gm = regression_metrics(ygap, best_individual_gap)
best_individual_tm = regression_metrics(ytemp, best_individual_temp)

champion_candidates["BEST_INDIVIDUAL"] = {
    "label": best_individual_name,
    "type": "INDIVIDUAL",
    "score": selection_score(best_individual_gm, best_individual_tm),
    "gap_metrics": best_individual_gm,
    "temporal_metrics": best_individual_tm,
    "components": [best_individual_name],
    "weights": [1.0],
}

# B) Best validation-tuned top-3 ensemble
best_ensemble_objective = ensemble_table.index[0]
best_ensemble = ensemble_results[best_ensemble_objective]

best_ensemble_gm = regression_metrics(
    ygap,
    best_ensemble["gap_pred"]
)
best_ensemble_tm = regression_metrics(
    ytemp,
    best_ensemble["temp_pred"]
)

champion_candidates["TOP3_ENSEMBLE"] = {
    "label": " + ".join(top_models),
    "type": "TOP3_ENSEMBLE",
    "score": selection_score(best_ensemble_gm, best_ensemble_tm),
    "gap_metrics": best_ensemble_gm,
    "temporal_metrics": best_ensemble_tm,
    "components": list(top_models),
    "weights": [float(w) for w in best_ensemble["weights"]],
}

# C) Tuned multi-seed Deep model
if RUN_MULTI_SEED_FINAL and "seed_ensemble_gap" in globals():
    multi_gap = regression_metrics(ygap, seed_ensemble_gap)
    multi_temp = regression_metrics(ytemp, seed_ensemble_temp)

    champion_candidates["MULTI_SEED_DEEP"] = {
        "label": f"{selected_arch} Multi-Seed ({len(MULTI_SEEDS)} seeds)",
        "type": "MULTI_SEED_DEEP",
        "score": selection_score(multi_gap, multi_temp),
        "gap_metrics": multi_gap,
        "temporal_metrics": multi_temp,
        "components": [selected_arch],
        "weights": [1.0],
    }

# D) Deep + profile + neighbor
if "best_final_blend" in globals() and best_final_blend.get("weights") is not None:
    blend_gap = regression_metrics(
        ygap,
        best_final_blend["gap_pred"]
    )
    blend_temp = regression_metrics(
        ytemp,
        best_final_blend["temp_pred"]
    )

    champion_candidates["DEEP_PROFILE_NEIGHBOR"] = {
        "label": f"{selected_arch} + Seasonal Profile + Neighbor",
        "type": "DEEP_PROFILE_NEIGHBOR",
        "score": selection_score(blend_gap, blend_temp),
        "gap_metrics": blend_gap,
        "temporal_metrics": blend_temp,
        "components": ["Deep","Profile","Neighbor"],
        "weights": [
            float(FINAL_COMPONENT_WEIGHTS["Deep"]),
            float(FINAL_COMPONENT_WEIGHTS["Profile"]),
            float(FINAL_COMPONENT_WEIGHTS["Neighbor"]),
        ],
    }

CHAMPION_KEY = min(
    champion_candidates,
    key=lambda k: champion_candidates[k]["score"]
)
CHAMPION = champion_candidates[CHAMPION_KEY]

champion_table = pd.DataFrame([
    {
        "Candidate": key,
        "Model / Strategy": value["label"],
        "SelectionScore": value["score"],
        "Gap_RMSE": value["gap_metrics"]["RMSE"],
        "Gap_MAE": value["gap_metrics"]["MAE"],
        "Gap_WMAPE_%": value["gap_metrics"]["WMAPE_%"],
        "Temporal_RMSE": value["temporal_metrics"]["RMSE"],
        "Temporal_MAE": value["temporal_metrics"]["MAE"],
        "Temporal_WMAPE_%": value["temporal_metrics"]["WMAPE_%"],
    }
    for key, value in champion_candidates.items()
]).set_index("Candidate").sort_values("SelectionScore")

display(champion_table.round(4))

# Communication-friendly derived index.
# IMPORTANT: this is NOT classification Accuracy.
gap_accuracy_index = max(
    0.0,
    100.0 - CHAMPION["gap_metrics"]["WMAPE_%"]
)
temp_accuracy_index = max(
    0.0,
    100.0 - CHAMPION["temporal_metrics"]["WMAPE_%"]
)

popup = f"""
<div style="
    border:4px solid #f4b400;
    border-radius:18px;
    padding:22px;
    background:linear-gradient(135deg,#082a5a,#104c97);
    color:white;
    font-family:Arial;
    box-shadow:0 8px 24px rgba(0,0,0,.25);
">
  <div style="font-size:18px;color:#ffd54f;font-weight:bold;">
    🏆 PEA AI 2026 — BEST FORECASTING MODEL
  </div>
  <div style="font-size:32px;font-weight:900;margin:8px 0 14px 0;">
    {CHAMPION["label"]}
  </div>
  <div style="font-size:16px;margin-bottom:12px;">
    Strategy: <b>{CHAMPION["type"]}</b> |
    Balanced Selection Score: <b>{CHAMPION["score"]:.4f}</b>
  </div>
  <table style="width:100%;color:white;font-size:15px;">
    <tr>
      <th style="text-align:left;">Validation</th>
      <th>RMSE (MW)</th>
      <th>MAE (MW)</th>
      <th>WMAPE</th>
      <th>Forecast Accuracy Index*</th>
    </tr>
    <tr>
      <td><b>Gap / Contest-Mimic</b></td>
      <td style="text-align:center;">{CHAMPION["gap_metrics"]["RMSE"]:.4f}</td>
      <td style="text-align:center;">{CHAMPION["gap_metrics"]["MAE"]:.4f}</td>
      <td style="text-align:center;">{CHAMPION["gap_metrics"]["WMAPE_%"]:.2f}%</td>
      <td style="text-align:center;font-size:20px;font-weight:bold;color:#7CFC00;">
        {gap_accuracy_index:.2f}%
      </td>
    </tr>
    <tr>
      <td><b>Temporal / Future</b></td>
      <td style="text-align:center;">{CHAMPION["temporal_metrics"]["RMSE"]:.4f}</td>
      <td style="text-align:center;">{CHAMPION["temporal_metrics"]["MAE"]:.4f}</td>
      <td style="text-align:center;">{CHAMPION["temporal_metrics"]["WMAPE_%"]:.2f}%</td>
      <td style="text-align:center;font-size:20px;font-weight:bold;color:#7CFC00;">
        {temp_accuracy_index:.2f}%
      </td>
    </tr>
  </table>
  <div style="font-size:12px;color:#d9e8ff;margin-top:10px;">
    *Forecast Accuracy Index = 100 − WMAPE.
    This is a derived communication index, not classification Accuracy.
  </div>
</div>
"""

display(HTML(popup))

print("CHAMPION_KEY:", CHAMPION_KEY)
print("CHAMPION:", CHAMPION["label"])
print("Components:", CHAMPION["components"])
print("Weights:", CHAMPION["weights"])


,Model / Strategy,SelectionScore,Gap_RMSE,Gap_MAE,Gap_WMAPE_%,Temporal_RMSE,Temporal_MAE,Temporal_WMAPE_%
Candidate,,,,,,,,
TOP3_ENSEMBLE,XGBoost + Ridge + LightGBM,0.9181,0.8931,0.4517,37.6688,0.8407,0.4230,39.8136
BEST_INDIVIDUAL,XGBoost,0.9325,0.9005,0.4553,37.9667,0.8740,0.4341,40.8569
DEEP_PROFILE_NEIGHBOR,MLP + Seasonal Profile + Neighbor,0.9344,0.9077,0.4444,37.0643,0.8958,0.4451,41.8919
MULTI_SEED_DEEP,MLP Multi-Seed (3 seeds),0.9362,0.9102,0.4462,37.2072,0.8930,0.4455,41.9323


Validation,RMSE (MW),MAE (MW),WMAPE,Forecast Accuracy Index*
Gap / Contest-Mimic,0.8931,0.4517,37.67%,62.33%
Temporal / Future,0.8407,0.4230,39.81%,60.19%


CHAMPION_KEY: TOP3_ENSEMBLE
CHAMPION: XGBoost + Ridge + LightGBM
Components: ['XGBoost', 'Ridge', 'LightGBM']
Weights: [0.45, 0.45, 0.10000000000000003]



## 19. Final refit and test prediction

Only **after** architecture/loss/optimizer/auxiliary decisions are frozen:

1. rebuild leakage-safe features from all known training rows,
2. train the selected deep configuration for 50 epochs,
3. create test predictions using the known train context,
4. save the submission.

### Test-label protection
If the supplied `vspp_test.csv` happens to contain `mw`, it is **dropped from prediction input** and ignored unless `ALLOW_TEST_LABEL_AUDIT=True`.

This separates genuine model selection from post-hoc auditing.


In [29]:

# ============================================================
# 25. Final feature rebuild for all known training data
# ============================================================
# Test features can use both directions because the supplied test timestamps
# are isolated gaps inside the known training period.
final_train_ctx = leakage_free_training_features(
    train_all,use_future_context=True
)

# Preserve exact original test/submission row order through context engineering.
test_features_input = test.drop(columns=["mw"],errors="ignore").copy()
test_features_input["_orig_order"] = np.arange(len(test_features_input))

# context_features does not require target MW to exist.
test_ctx = context_features(
    test_features_input,
    train_all,
    use_future_context=True
)
test_ctx = test_ctx.sort_values("_orig_order").reset_index(drop=True)

assert np.array_equal(
    test_ctx["_orig_order"].to_numpy(),
    np.arange(len(test_ctx))
)

print(final_train_ctx.shape,test_ctx.shape)


(42129, 69) (1719, 69)


In [30]:

# ============================================================
# 26. Rebuild sequence normalization for final training/test
# ============================================================
final_train_samples_raw = make_day_samples(final_train_ctx)

# make_day_samples expects an 'mw' column for targets.
# Create a placeholder only for test sequence structure; the values are never used for fitting.
test_seq_df = test_ctx.copy()
test_seq_df["mw"] = np.nan
test_samples_raw = make_day_samples(test_seq_df)

final_all_X = np.concatenate([s["X"] for s in final_train_samples_raw],axis=0)
final_weather_mean = final_all_X[:,WEATHER_START:WEATHER_END].mean(axis=0)
final_weather_std = final_all_X[:,WEATHER_START:WEATHER_END].std(axis=0)+1e-6

def normalize_final(samples):
    out = []
    for s in samples:
        z = copy.deepcopy(s)
        X = z["X"].copy()
        X[:,:GEN_N] /= z["scale"]
        X[:,WEATHER_START:WEATHER_END] = (
            X[:,WEATHER_START:WEATHER_END]-final_weather_mean
        )/final_weather_std
        z["X"] = np.nan_to_num(X,nan=0.0,posinf=0.0,neginf=0.0).astype(np.float32)
        out.append(z)
    return out

final_train_samples = normalize_final(final_train_samples_raw)
final_test_samples = normalize_final(test_samples_raw)

final_loader = tud.DataLoader(
    DayDataset(final_train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True
)


In [31]:
# ============================================================
# 27. Final full-data refit helpers
# ============================================================
def train_final_deep(
    architecture,
    seed,
    loss_name="Composite",
    optimizer_name="AdamW",
    aux_mode="NONE",
    tcn_hidden=128,
    tcn_dropout=0.15562200257933947,
    lr=None,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    n_features = final_train_samples[0]["X"].shape[1]
    model = build_model(
        architecture,
        n_features,
        tcn_hidden=tcn_hidden,
        tcn_dropout=tcn_dropout
    ).to(DEVICE)

    opt = make_optimizer(
        optimizer_name,
        model.parameters(),
        lr=lr
    )

    for epoch in range(1, EPOCHS + 1):
        model.train()

        for X,y,mask,base,scale,ramp_class,ramp_binary,ramp_mask in final_loader:
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)
            base = base.to(DEVICE)
            scale = scale.to(DEVICE)
            ramp_class = ramp_class.to(DEVICE)
            ramp_binary = ramp_binary.to(DEVICE)
            ramp_mask = ramp_mask.to(DEVICE)

            opt.zero_grad()

            residual_pred, class_logits, binary_logit = model(X)
            pred_mw = base + residual_pred * scale[:, None]

            reg = regression_loss(
                pred_mw,
                y,
                mask,
                loss_name
            )
            total = reg

            if aux_mode != "NONE":
                class_mode = "CCE" if aux_mode == "CCE_BCE" else "HINGE"
                multi, bce = auxiliary_loss(
                    class_logits,
                    binary_logit,
                    ramp_class,
                    ramp_binary,
                    ramp_mask,
                    class_mode
                )
                total = reg + 0.02 * multi + 0.02 * bce

            total.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()

    return model

def predict_test_with_deep_model(model):
    model.eval()
    row_map = {}

    with torch.no_grad():
        for s in final_test_samples:
            X = torch.tensor(
                s["X"],
                dtype=torch.float32,
                device=DEVICE
            )[None]

            res, _, _ = model(X)
            mw = s["base"] + res.cpu().numpy()[0] * s["scale"]

            for q, val in enumerate(mw):
                row_map[(s["date"], s["site"], q)] = float(val)

    pred = np.array([
        row_map[
            (r["generation_date"], r["name"], int(r["q"]))
        ]
        for _, r in test_ctx.iterrows()
    ])

    return physical_postprocess(
        pred,
        test_ctx,
        train_all
    )

# Full-data matrices for any classical champion component.
Xfinal_tab, Xtest_tab = prepare_tabular(
    final_train_ctx,
    test_ctx
)
yfinal_tab = final_train_ctx["mw"].to_numpy()

_final_component_cache = {}

def final_component_prediction(model_name):
    """Refit one component on ALL known training rows and predict test rows."""
    if model_name in _final_component_cache:
        return _final_component_cache[model_name]

    if model_name == "Seasonal_Profile_Median":
        pred = test_ctx["profile_median"].fillna(0).to_numpy()
        pred = physical_postprocess(
            pred,
            test_ctx,
            train_all
        )

    elif model_name == "Previous_Day_Persistence":
        pred = test_ctx["prev_1d"].fillna(
            test_ctx["profile_median"]
        ).to_numpy()
        pred = physical_postprocess(
            pred,
            test_ctx,
            train_all
        )

    elif model_name in CLASSICAL_MODEL_NAMES:
        print("Final full-data refit:", model_name)

        model = build_classical_model(model_name)
        model.fit(Xfinal_tab, yfinal_tab)

        pred = model.predict(Xtest_tab)
        pred = physical_postprocess(
            pred,
            test_ctx,
            train_all
        )

    elif model_name in DEEP_ARCHITECTURES:
        print("Final full-data Deep refit:", model_name)

        lr = (
            0.0018974773610953666
            if model_name == "TCN"
            else None
        )

        model = train_final_deep(
            architecture=model_name,
            seed=SEED,
            loss_name="Composite",
            optimizer_name="AdamW",
            aux_mode="NONE",
            lr=lr
        )

        pred = predict_test_with_deep_model(model)

    else:
        raise ValueError(
            f"Cannot refit final component: {model_name}"
        )

    _final_component_cache[model_name] = pred
    return pred

print("Final refit helpers ready.")


Final refit helpers ready.


In [32]:
# ============================================================
# 28. Refit the ACTUAL CHAMPION + create final submission
# ============================================================
def fit_selected_multiseed_deep():
    preds = []

    for sd in MULTI_SEEDS:
        print("Final selected Deep seed:", sd)

        model = train_final_deep(
            architecture=selected_arch,
            seed=sd,
            loss_name=selected_loss,
            optimizer_name=selected_optimizer,
            aux_mode=selected_aux,
        )

        preds.append(
            predict_test_with_deep_model(model)
        )

    return np.mean(
        np.vstack(preds),
        axis=0
    )

if CHAMPION["type"] == "INDIVIDUAL":
    final_test_pred = final_component_prediction(
        CHAMPION["components"][0]
    )

elif CHAMPION["type"] == "TOP3_ENSEMBLE":
    weighted_predictions = []

    for model_name, weight in zip(
        CHAMPION["components"],
        CHAMPION["weights"]
    ):
        print(
            f"Final ensemble component: {model_name} "
            f"(weight={weight:.2f})"
        )

        component_pred = final_component_prediction(
            model_name
        )
        weighted_predictions.append(
            weight * component_pred
        )

    final_test_pred = np.sum(
        np.vstack(weighted_predictions),
        axis=0
    )

    final_test_pred = physical_postprocess(
        final_test_pred,
        test_ctx,
        train_all
    )

elif CHAMPION["type"] == "MULTI_SEED_DEEP":
    final_test_pred = fit_selected_multiseed_deep()

    final_test_pred = physical_postprocess(
        final_test_pred,
        test_ctx,
        train_all
    )

elif CHAMPION["type"] == "DEEP_PROFILE_NEIGHBOR":
    deep_test_mean = fit_selected_multiseed_deep()

    profile_test = (
        test_ctx["profile_median"]
        .fillna(0)
        .to_numpy()
    )

    neighbor_test = (
        test_ctx[["prev_1d","next_1d"]]
        .mean(axis=1)
        .fillna(test_ctx["profile_median"])
        .to_numpy()
    )

    wd, wp, wn = CHAMPION["weights"]

    final_test_pred = (
        wd * deep_test_mean
        + wp * profile_test
        + wn * neighbor_test
    )

    final_test_pred = physical_postprocess(
        final_test_pred,
        test_ctx,
        train_all
    )

else:
    raise ValueError(
        f"Unknown champion type: {CHAMPION['type']}"
    )

submission_cols = [
    c
    for c in [
        "pred_id",
        "generation_date",
        "generation_time",
        "name"
    ]
    if c in test.columns
]

submission = test[submission_cols].copy()
submission["mw"] = final_test_pred

if "pred_id" in submission.columns and "pred_id" in test.columns:
    assert np.array_equal(
        submission["pred_id"].to_numpy(),
        test["pred_id"].to_numpy()
    )

SUBMISSION_FILE = (
    f"PEA_AI_2026_VSPP_submission_v4_"
    f"{CHAMPION_KEY}.csv"
)

submission.to_csv(
    SUBMISSION_FILE,
    index=False
)

display(submission.head())

print("Final champion:", CHAMPION["label"])
print("Champion type:", CHAMPION["type"])
print("submission rows:", len(submission))
print("saved:", SUBMISSION_FILE)


Final ensemble component: XGBoost (weight=0.45)
Final full-data refit: XGBoost
Final ensemble component: Ridge (weight=0.45)
Final full-data refit: Ridge
Final ensemble component: LightGBM (weight=0.10)
Final full-data refit: LightGBM


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,pred_id,generation_date,generation_time,name,mw
0,6039,2026-06-08,00:00,ZS11,0.0
1,6040,2026-06-08,00:00,ZS12,0.0
2,6041,2026-06-08,00:00,ZS13,0.0
3,6042,2026-06-08,00:00,ZS14,0.0
4,6043,2026-06-08,00:00,ZS22,0.0


Final champion: XGBoost + Ridge + LightGBM
Champion type: TOP3_ENSEMBLE
submission rows: 1719
saved: PEA_AI_2026_VSPP_submission_v4_TOP3_ENSEMBLE.csv


In [33]:

# ============================================================
# 29. OPTIONAL post-model test audit
# ============================================================
if ALLOW_TEST_LABEL_AUDIT and "mw" in test.columns:
    audit_metrics = regression_metrics(
        test["mw"].to_numpy(),
        final_test_pred
    )
    print("POST-MODEL TEST AUDIT ONLY:",audit_metrics)

    audit = test.copy().rename(columns={"mw":"actual_mw"})
    audit["predicted_mw"] = final_test_pred
    audit["abs_error"] = np.abs(audit["actual_mw"]-audit["predicted_mw"])
    audit.to_csv("PEA_AI_2026_VSPP_test_audit_v4.csv",index=False)
else:
    print(
        "Test labels remain locked. "
        "Model selection is based only on the defined validation sets."
    )


Test labels remain locked. Model selection is based only on the defined validation sets.



## 20. Contest interpretation

### Regression winner
Use:
1. official PEA metric if specified;
2. otherwise lowest **Gap RMSE** with acceptable temporal robustness;
3. Pareto RMSE/MAE/WMAPE;
4. bootstrap confidence intervals.

### Ramp-event winner
Use:
- Accuracy,
- Recall,
- F1.

### Recommended technical claim
> “The final VSPP forecasting model was selected through a leakage-controlled benchmark of statistical, machine-learning and seven deep-learning architectures. Every deep model was trained for 50 epochs under the same validation protocol. Regression performance was measured using RMSE, MAE, zero-safe percentage metrics and site-level robustness, while BCE/CCE/hinge and Accuracy/Recall/F1 were separately applied to renewable ramp-event detection. The final configuration was refit on all known training data only after model selection was frozen.”

### Why this is stronger than forcing TCN to win
The contest contribution becomes **a model-selection and grid-intelligence framework**, not merely one neural network.


In [34]:
# ============================================================
# Forecasting scorecard — regression vs classification Accuracy
# ============================================================
forecast_scorecard = pd.DataFrame({
    "Validation": [
        "Gap / Contest-Mimic",
        "Temporal / Future"
    ],
    "RMSE_MW": [
        CHAMPION["gap_metrics"]["RMSE"],
        CHAMPION["temporal_metrics"]["RMSE"],
    ],
    "MAE_MW": [
        CHAMPION["gap_metrics"]["MAE"],
        CHAMPION["temporal_metrics"]["MAE"],
    ],
    "WMAPE_pct": [
        CHAMPION["gap_metrics"]["WMAPE_%"],
        CHAMPION["temporal_metrics"]["WMAPE_%"],
    ],
})

# Communication-only derived index:
# 100 - WMAPE. NOT classification Accuracy.
forecast_scorecard["Forecast_Accuracy_Index_pct"] = (
    100.0 - forecast_scorecard["WMAPE_pct"]
).clip(lower=0.0)

display(
    forecast_scorecard.round(4)
)

if len(auxiliary_table):
    best_ramp_mode = (
        auxiliary_table["Gap_RampBinary_F1"]
        .idxmax()
    )

    print(
        "Best auxiliary ramp-classification mode by F1:",
        best_ramp_mode
    )
    print(
        "Ramp Accuracy / Recall / F1 are reported separately "
        "and are not used as continuous-MW forecasting Accuracy."
    )


,Validation,RMSE_MW,MAE_MW,WMAPE_pct,Forecast_Accuracy_Index_pct
0,Gap / Contest-Mimic,0.8931,0.4517,37.6688,62.3312
1,Temporal / Future,0.8407,0.4230,39.8136,60.1864


Best auxiliary ramp-classification mode by F1: HINGE_BCE
Ramp Accuracy / Recall / F1 are reported separately and are not used as continuous-MW forecasting Accuracy.


### Forecasting accuracy note

For continuous MW forecasting, the primary model-selection metrics are **RMSE, MAE and WMAPE**.

The notebook also displays:

**Forecast Accuracy Index = 100 − WMAPE**

as a communication-friendly derived index.

Accuracy / Recall / F1 remain reserved for the auxiliary renewable ramp-classification task.


In [35]:
# ============================================================
# 30. Export all experiment tables + champion metadata
# ============================================================
EXPORT_DIR = Path("pea_ai_2026_v4_outputs")
EXPORT_DIR.mkdir(exist_ok=True)

classical_table.to_csv(
    EXPORT_DIR / "01_classical_table.csv"
)

if len(deep_table):
    deep_table.to_csv(
        EXPORT_DIR / "02_deep_architecture_table.csv"
    )

if len(loss_table):
    loss_table.to_csv(
        EXPORT_DIR / "03_loss_table.csv"
    )

if len(optimizer_table):
    optimizer_table.to_csv(
        EXPORT_DIR / "04_optimizer_table.csv"
    )

if len(auxiliary_table):
    auxiliary_table.to_csv(
        EXPORT_DIR / "05_auxiliary_ramp_table.csv"
    )

scoreboard.to_csv(
    EXPORT_DIR / "06_unified_scoreboard.csv"
)

gap_pareto.to_csv(
    EXPORT_DIR / "07_gap_pareto.csv"
)

ensemble_table.to_csv(
    EXPORT_DIR / "08_ensemble_table.csv"
)

champion_table.to_csv(
    EXPORT_DIR / "09_champion_candidates.csv"
)

forecast_scorecard.to_csv(
    EXPORT_DIR / "10_forecasting_scorecard.csv",
    index=False
)

champion_metadata = {
    "dataset_folder_name": DATASET_FOLDER_NAME,
    "dataset_folder_id": DATASET_FOLDER_ID,
    "dataset_folder_url": DATASET_FOLDER_URL,
    "champion_key": CHAMPION_KEY,
    "champion_label": CHAMPION["label"],
    "champion_type": CHAMPION["type"],
    "selection_score": float(CHAMPION["score"]),
    "components": list(CHAMPION["components"]),
    "weights": [
        float(w)
        for w in CHAMPION["weights"]
    ],
    "gap_metrics": {
        k: float(v)
        for k, v in CHAMPION["gap_metrics"].items()
        if pd.notna(v)
    },
    "temporal_metrics": {
        k: float(v)
        for k, v in CHAMPION["temporal_metrics"].items()
        if pd.notna(v)
    },
    "metric_weights": {
        "RMSE": RMSE_WEIGHT,
        "MAE": MAE_WEIGHT,
        "WMAPE": WMAPE_WEIGHT,
    },
    "validation_weights": {
        "Gap": GAP_WEIGHT,
        "Temporal": TEMPORAL_WEIGHT,
    },
}

with open(
    EXPORT_DIR / "11_champion_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        champion_metadata,
        f,
        indent=2
    )

submission.to_csv(
    EXPORT_DIR / "12_final_submission.csv",
    index=False
)

print(
    "Exported v4 result package to:",
    EXPORT_DIR.resolve()
)


Exported v4 result package to: /content/pea_ai_2026_v4_outputs


In [ ]:
!zip -r pea_ai_2026_v4_outputs.zip ./pea_ai_2026_v4_outputs


The `pea_ai_2026_v4_outputs.zip` package contains the unified scoreboard, champion comparison, forecasting scorecard, champion metadata, and final submission.
